### Version 20

# NIFTY-50 Multi-Horizon Stock Predictor - Training Pipeline v2.0## 📋 OverviewThis notebook implements a sophisticated **ensemble deep learning system** for predicting stock returns across multiple time horizons (1-180 days). The model combines LSTM networks with multi-head attention mechanisms and incorporates both technical indicators and market context features.---## 🏗️ Architecture Components### 1. **Model Architecture**- **Base Model**: LSTM (256 → 128 units) with Multi-Head Attention (8 heads)- **Regularization**: Dropout (0.2-0.3), Batch Normalization, Layer Normalization- **Output**: 180 values (daily return predictions for next 180 days)### 2. **Ensemble Strategy**- **Number of Models**: 5 independent models with different random seeds- **Weighting**: Performance-based (inverse validation loss)- **Benefit**: Reduces overfitting, provides uncertainty estimates### 3. **Custom Loss Function**- **DirectionAwareLoss**: Combines MSE with directional accuracy penalty- **Formula**: `(1-α) × MSE + α × DirectionPenalty`- **Weight (α)**: 0.3 (balances magnitude and direction)---## 📊 Feature Engineering### Stock-Specific Technical Indicators (9 features)| Indicator | Purpose ||-----------|---------|| **RSI** | Momentum oscillator (overbought/oversold) || **ADX** | Trend strength measurement || **NATR** | Normalized volatility || **OBV_Slope** | Volume-price relationship || **Dist_SMA** | Distance from 50-day moving average || **MACD** | Trend following momentum || **ROC** | Rate of change || **Vol_Ratio** | Volume relative to 20-day average || **BB_Position** | Position within Bollinger Bands |### Market Context Features (6 features)| Feature | Purpose ||---------|---------|| **Market_Strength** | NIFTY position vs MA(20, 50, 200) || **Market_Momentum** | NIFTY 20-day return || **Market_Volatility** | Annualized rolling volatility || **Market_Trend** | Sharpe-like trend metric || **Relative_Strength** | Stock return vs NIFTY return || **Beta** | 60-day rolling beta to NIFTY |**Total Features**: 15---## 🔄 Training Pipeline### Phase 1: Data Collection- Downloads 25 years of historical data for 19 NIFTY-50 stocks- Downloads NIFTY index for market context- Handles missing data and ensures data quality### Phase 2: Feature Calculation- Computes all 15 technical and market features- Aligns stock data with NIFTY index dates- Removes NaN values from indicator calculations### Phase 3: Sequence Creation- **Sequence Length**: 60 days (input window)- **Target Horizon**: 180 days (output predictions)- Creates ~76,000 training sequences across all stocks- **Shape**: X=(75980, 60, 15), y=(75980, 180)### Phase 4: Data Preprocessing- **Scaling**: StandardScaler fitted on 80% of data (prevents leakage)- **Train/Val Split**: 80/20 with 180-day gap to prevent lookahead bias- **Gap Purpose**: Ensures validation targets don't overlap with training features### Phase 5: Model Training- **Ensemble**: Trains 5 models with seeds [42, 43, 44, 45, 46]- **Batch Size**: 1024 (efficient GPU utilization)- **Epochs**: 50 with early stopping (patience=7)- **Optimizer**: Adam (lr=0.001) with ReduceLROnPlateau- **Callbacks**: ModelCheckpoint, EarlyStopping, ReduceLROnPlateau### Phase 6: Evaluation- Calculates metrics across multiple horizons: [1, 5, 10, 20, 60, 120, 180 days]- **Metrics**: Direction Accuracy, Correlation, MAE, RMSE- Saves all metrics and model artifacts---## 📈 Expected PerformanceBased on validation set:| Horizon | Direction Accuracy | Correlation | MAE | RMSE ||---------|-------------------|-------------|-----|------|| **20 days** | 56.9% | +0.196 | 6.40% | 8.33% || **60 days** | 59.8% | +0.319 | 11.42% | 14.47% || **120 days** | 64.1% | +0.437 | 15.92% | 20.41% || **180 days** | 67.0% | +0.480 | 19.44% | 25.49% |**Key Insight**: Model performs better at longer horizons (trend following)---## 💾 Output FilesAfter training completes, the following files are generated:1. **`scaler_v2.pkl`** - StandardScaler for feature normalization2. **`metrics_v2.pkl`** - Complete evaluation metrics3. **`ensemble_models/`** - Directory containing 5 trained models   - `model_1.keras` through `model_5.keras`4. **`nifty50_integrated_ensemble_info.pkl`** - Ensemble weights and metadata---## ⚙️ ConfigurationKey hyperparameters (can be modified in CONFIG dict):```pythonCONFIG = {    'SEQ_LENGTH': 60,          # Input sequence length    'MAX_HORIZON': 180,        # Maximum prediction horizon    'TRAIN_YEARS': 25,         # Historical data period    'BATCH_SIZE': 1024,        # Training batch size    'EPOCHS': 50,              # Maximum epochs    'N_ENSEMBLE': 5,           # Number of ensemble models    'ENABLE_ENSEMBLE': True,   # Use ensemble vs single model    'ENABLE_MARKET_FEATURES': True,   # Include market context    'ENABLE_CUSTOM_LOSS': True        # Use DirectionAwareLoss}```---## 🚀 UsageSimply run all cells in order. The training process will:1. ✅ Download and process data (~5-10 minutes)2. ✅ Train ensemble models (~30-60 minutes on GPU)3. ✅ Evaluate and save results**Total Runtime**: ~45-70 minutes (depends on hardware)---## 🔍 Key Design Decisions1. **Why Ensemble?** Reduces variance, provides uncertainty estimates, improves robustness2. **Why Custom Loss?** Financial applications care about direction, not just magnitude3. **Why Market Features?** Stocks don't move in isolation; market regime matters4. **Why 60-day Sequence?** Balances short-term patterns with long-term trends (~3 months)5. **Why 180-day Gap?** Prevents data leakage in validation set---## ⚠️ Important Notes- **Data Leakage Prevention**: Scaler fitted only on training data, 180-day gap in validation- **Computational Requirements**: GPU recommended (CPU training will be 10-20x slower)- **Memory Usage**: ~8-16GB RAM required for full dataset- **Reproducibility**: Random seeds set for numpy and tensorflow

In [5]:
"""
NIFTY-50 Multi-Horizon Stock Predictor - Complete Integrated Training System v2.0

Integrates all improvements:
- Market context features
- Custom loss functions
- Model ensemble
- Enhanced evaluation
- Proper data handling

Author: Enhanced Training Pipeline
Date: 2025
"""

import yfinance as yf
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import os
import warnings
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout, 
                                      BatchNormalization, MultiHeadAttention, 
                                      LayerNormalization, Add)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import backend as K

warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION
# ==============================================================================
CONFIG = {
    'SEQ_LENGTH': 60,
    'MAX_HORIZON': 180,
    'TRAIN_YEARS': 25,
    'BATCH_SIZE': 1024,
    'EPOCHS': 50,
    'N_ENSEMBLE': 5,
    'ENABLE_ENSEMBLE': True,
    'ENABLE_MARKET_FEATURES': True,
    'ENABLE_CUSTOM_LOSS': True,
    
    'MODEL_PREFIX': 'nifty50_integrated',
    'SCALER_NAME': 'scaler_v2.pkl',
    'METRICS_NAME': 'metrics_v2.pkl',
    'ENSEMBLE_DIR': 'ensemble_models',
    
    'TICKERS': [
        'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
        'HINDUNILVR.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'ITC.NS', 'KOTAKBANK.NS',
        'LICI.NS', 'LT.NS', 'AXISBANK.NS', 'HCLTECH.NS', 'ASIANPAINT.NS',
        'MARUTI.NS', 'SUNPHARMA.NS', 'TITAN.NS', 'ULTRACEMCO.NS'
    ]
}

# ==============================================================================
# CUSTOM LOSS FUNCTIONS
# ==============================================================================

class DirectionAwareLoss(tf.keras.losses.Loss):
    """Penalize wrong direction predictions more"""
    def __init__(self, direction_weight=0.3, name='direction_aware_loss'):
        super().__init__(name=name)
        self.direction_weight = direction_weight
        
    def call(self, y_true, y_pred):
        # MSE component
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        
        # Direction penalty
        direction_penalty = tf.reduce_mean(
            tf.cast(tf.sign(y_true) != tf.sign(y_pred), tf.float32)
        )
        
        return (1 - self.direction_weight) * mse + self.direction_weight * direction_penalty * 100

# ==============================================================================
# ENHANCED INDICATORS
# ==============================================================================

class EnhancedIndicators:
    @staticmethod
    def get_rsi(series, period=14):
        delta = series.diff()
        gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
        loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))

    @staticmethod
    def get_adx(high, low, close, period=14):
        plus_dm = high.diff()
        minus_dm = low.diff()
        plus_dm[plus_dm < 0] = 0
        minus_dm[minus_dm > 0] = 0
        tr1 = pd.DataFrame(high - low)
        tr2 = pd.DataFrame(abs(high - close.shift(1)))
        tr3 = pd.DataFrame(abs(low - close.shift(1)))
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1/period, adjust=False).mean()
        plus_di = 100 * (plus_dm.ewm(alpha=1/period, adjust=False).mean() / atr)
        minus_di = 100 * (abs(minus_dm).ewm(alpha=1/period, adjust=False).mean() / atr)
        dx = (abs(plus_di - minus_di) / (plus_di + minus_di)) * 100
        return dx.ewm(alpha=1/period, adjust=False).mean()

    @staticmethod
    def get_natr(high, low, close, period=14):
        tr1 = high - low
        tr2 = abs(high - close.shift(1))
        tr3 = abs(low - close.shift(1))
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1/period, adjust=False).mean()
        return (atr / close) * 100

    @staticmethod
    def get_obv_slope(close, volume, period=14):
        obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
        return obv.diff(period)

    @staticmethod
    def get_dist_sma(close, period=50):
        sma = close.rolling(period).mean()
        return (close - sma) / sma
    
    @staticmethod
    def get_macd(close, fast=12, slow=26, signal=9):
        ema_fast = close.ewm(span=fast, adjust=False).mean()
        ema_slow = close.ewm(span=slow, adjust=False).mean()
        macd = ema_fast - ema_slow
        macd_signal = macd.ewm(span=signal, adjust=False).mean()
        return macd - macd_signal
    
    @staticmethod
    def get_roc(close, period=10):
        return ((close - close.shift(period)) / close.shift(period)) * 100
    
    @staticmethod
    def get_volume_ratio(volume, period=20):
        vol_ma = volume.rolling(period).mean()
        return volume / vol_ma
    
    @staticmethod
    def get_bollinger_position(close, period=20, std=2):
        sma = close.rolling(period).mean()
        rolling_std = close.rolling(period).std()
        upper = sma + (rolling_std * std)
        lower = sma - (rolling_std * std)
        return (close - lower) / (upper - lower)

# ==============================================================================
# MARKET CONTEXT FEATURES
# ==============================================================================

class MarketContextFeatures:
    """Add market-level features to improve predictions"""
    
    @staticmethod
    def calculate_market_features(nifty_close):
        """Calculate all market context features"""
        features = pd.DataFrame(index=nifty_close.index)
        
        # Market strength (position vs MAs)
        for period in [20, 50, 200]:
            ma = nifty_close.rolling(period).mean()
            features[f'above_ma_{period}'] = (nifty_close > ma).astype(float)
        
        features['market_strength'] = features[[f'above_ma_{p}' for p in [20, 50, 200]]].mean(axis=1)
        
        # Market momentum
        features['market_momentum'] = nifty_close.pct_change(20) * 100
        
        # Market volatility
        returns = nifty_close.pct_change()
        features['market_volatility'] = returns.rolling(20).std() * np.sqrt(252) * 100
        
        # Trend strength
        features['market_trend'] = (
            returns.rolling(20).mean() / (returns.rolling(20).std() + 1e-8)
        )
        
        return features[['market_strength', 'market_momentum', 'market_volatility', 'market_trend']]

# ==============================================================================
# DATA PREPARATION
# ==============================================================================

def prepare_stock_features(ticker, nifty_df):
    """
    Prepare all features for a single stock including market context.
    
    Returns: DataFrame with all features, or None if insufficient data
    """
    print(f"  → Processing {ticker}...")
    
    try:
        # Download stock data
        df = yf.download(ticker, period=f"{CONFIG['TRAIN_YEARS']}y", interval="1d", progress=False)
        if len(df) < 500:
            print(f"    ⚠️  Insufficient data")
            return None
            
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        
        # === STOCK-SPECIFIC INDICATORS ===
        df['RSI'] = EnhancedIndicators.get_rsi(df['Close'])
        df['ADX'] = EnhancedIndicators.get_adx(df['High'], df['Low'], df['Close'])
        df['NATR'] = EnhancedIndicators.get_natr(df['High'], df['Low'], df['Close'])
        df['OBV_Slope'] = EnhancedIndicators.get_obv_slope(df['Close'], df['Volume'])
        df['Dist_SMA'] = EnhancedIndicators.get_dist_sma(df['Close'])
        df['MACD'] = EnhancedIndicators.get_macd(df['Close'])
        df['ROC'] = EnhancedIndicators.get_roc(df['Close'])
        df['Vol_Ratio'] = EnhancedIndicators.get_volume_ratio(df['Volume'])
        df['BB_Position'] = EnhancedIndicators.get_bollinger_position(df['Close'])
        
        # === MARKET CONTEXT FEATURES ===
        if CONFIG['ENABLE_MARKET_FEATURES'] and nifty_df is not None:
            # Align NIFTY to stock dates
            nifty_aligned = nifty_df.reindex(df.index, method='ffill')
            
            # Add market features
            market_features = MarketContextFeatures.calculate_market_features(nifty_aligned['Close'])
            df = pd.concat([df, market_features], axis=1)
            
            # Relative strength vs market
            stock_returns = df['Close'].pct_change(20)
            market_returns = nifty_aligned['Close'].pct_change(20)
            df['relative_strength'] = stock_returns - market_returns
            
            # Beta to market
            stock_ret = df['Close'].pct_change()
            market_ret = nifty_aligned['Close'].pct_change()
            df['beta'] = (
                stock_ret.rolling(60).cov(market_ret) / 
                (market_ret.rolling(60).var() + 1e-8)
            )
        
        # Drop NaN from indicators
        df.dropna(inplace=True)
        
        if len(df) < CONFIG['SEQ_LENGTH'] + 100:
            print(f"    ⚠️  Insufficient data after cleaning")
            return None
        
        print(f"    ✓ {len(df)} samples")
        return df
        
    except Exception as e:
        print(f"    ❌ Error: {e}")
        return None


def calculate_targets(df):
    """Calculate future returns for all horizons"""
    target_data = {}
    for i in range(1, CONFIG['MAX_HORIZON'] + 1):
        col_name = f'Target_Day_{i}'
        target_data[col_name] = df['Close'].pct_change(periods=i).shift(-i) * 100
    
    targets_df = pd.DataFrame(target_data, index=df.index)
    return targets_df


def create_sequences_safe(features, targets, seq_length):
    """Create sequences without boundary crossing"""
    if len(features) < seq_length + 1:
        return np.array([]), np.array([])
    
    X, y = [], []
    for i in range(len(features) - seq_length):
        X.append(features[i:(i + seq_length)])
        y.append(targets[i + seq_length])
    
    return np.array(X), np.array(y)


def get_feature_columns():
    """Return list of feature columns to use"""
    features = [
        'RSI', 'ADX', 'NATR', 'OBV_Slope', 'Dist_SMA',
        'MACD', 'ROC', 'Vol_Ratio', 'BB_Position'
    ]
    
    if CONFIG['ENABLE_MARKET_FEATURES']:
        features.extend([
            'market_strength', 'market_momentum', 'market_volatility', 'market_trend',
            'relative_strength', 'beta'
        ])
    
    return features

# ==============================================================================
# MODEL ARCHITECTURE
# ==============================================================================

def build_enhanced_model(seq_length, n_features, n_outputs, seed=42):
    """Build enhanced model with attention mechanism"""
    
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
    input_layer = Input(shape=(seq_length, n_features))
    
    # First LSTM layer
    x = LSTM(256, return_sequences=True)(input_layer)
    x = Dropout(0.3)(x)
    x = BatchNormalization()(x)
    
    # Multi-head attention
    attn_output = MultiHeadAttention(
        num_heads=8, 
        key_dim=32,
        dropout=0.2
    )(x, x)
    
    # Residual connection
    x = Add()([x, attn_output])
    x = LayerNormalization()(x)
    
    # Second LSTM layer
    x = LSTM(128, return_sequences=False)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Dense layers
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(128, activation='relu')(x)
    
    # Output layer
    output_layer = Dense(n_outputs, activation='linear')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    
    return model

# ==============================================================================
# ENSEMBLE TRAINING
# ==============================================================================

class ModelEnsemble:
    """Train and manage ensemble of models"""
    
    def __init__(self, n_models=5):
        self.n_models = n_models
        self.models = []
        self.weights = []
    
    def train(self, X_train, y_train, X_val, y_val, seq_length, n_features, n_outputs):
        """Train ensemble of diverse models"""
        
        print(f"\n{'='*80}")
        print(f"TRAINING ENSEMBLE OF {self.n_models} MODELS")
        print(f"{'='*80}\n")
        
        os.makedirs(CONFIG['ENSEMBLE_DIR'], exist_ok=True)
        
        for i in range(self.n_models):
            print(f"\n--- Model {i+1}/{self.n_models} (Seed: {42+i}) ---")
            
            # Build model with different seed
            model = build_enhanced_model(seq_length, n_features, n_outputs, seed=42+i)
            
            # Compile with custom loss if enabled
            if CONFIG['ENABLE_CUSTOM_LOSS']:
                loss = DirectionAwareLoss(direction_weight=0.3)
            else:
                loss = 'mse'
            
            model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                loss=loss,
                metrics=['mae']
            )
            
            # Callbacks
            model_path = os.path.join(CONFIG['ENSEMBLE_DIR'], f'model_{i+1}.keras')
            callbacks = [
                EarlyStopping(
                    monitor='val_loss',
                    patience=7,
                    restore_best_weights=True,
                    verbose=0
                ),
                ModelCheckpoint(
                    model_path,
                    save_best_only=True,
                    monitor='val_loss',
                    verbose=0
                ),
                ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.5,
                    patience=3,
                    min_lr=1e-6,
                    verbose=0
                )
            ]
            
            # Train with data shuffling
            indices = np.random.RandomState(42+i).permutation(len(X_train))
            X_shuffled = X_train[indices]
            y_shuffled = y_train[indices]
            
            history = model.fit(
                X_shuffled, y_shuffled,
                validation_data=(X_val, y_val),
                epochs=CONFIG['EPOCHS'],
                batch_size=CONFIG['BATCH_SIZE'],
                callbacks=callbacks,
                verbose=0
            )
            
            # Calculate model weight based on validation performance
            val_loss = min(history.history['val_loss'])
            weight = 1.0 / (val_loss + 1e-6)
            
            self.models.append(model)
            self.weights.append(weight)
            
            print(f"   Best Val Loss: {val_loss:.4f} | Weight: {weight:.2f}")
        
        # Normalize weights
        self.weights = np.array(self.weights)
        self.weights = self.weights / self.weights.sum()
        
        print(f"\n✅ Ensemble complete!")
        print(f"   Normalized weights: {[f'{w:.3f}' for w in self.weights]}")
    
    def predict(self, X):
        """Weighted average prediction"""
        predictions = []
        for model in self.models:
            pred = model.predict(X, verbose=0)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        
        return weighted_pred
    
    def predict_with_uncertainty(self, X):
        """Return mean and std across ensemble"""
        predictions = []
        for model in self.models:
            pred = model.predict(X, verbose=0)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        mean_pred = np.average(predictions, axis=0, weights=self.weights)
        std_pred = np.std(predictions, axis=0)
        
        return mean_pred, std_pred

# ==============================================================================
# EVALUATION
# ==============================================================================

def evaluate_model(predictor, X_val, y_val, horizons=[1, 5, 10, 20, 60, 120, 180]):
    """Comprehensive evaluation across horizons"""
    
    print("\n" + "="*80)
    print("VALIDATION METRICS BY HORIZON")
    print("="*80)
    
    if CONFIG['ENABLE_ENSEMBLE']:
        predictions = predictor.predict(X_val)
    else:
        predictions = predictor.predict(X_val, verbose=0)
    
    metrics = {}
    
    for horizon in horizons:
        if horizon > y_val.shape[1]:
            continue
        
        actual = y_val[:, horizon-1]
        pred = predictions[:, horizon-1]
        
        # Direction accuracy
        direction_acc = np.mean(np.sign(actual) == np.sign(pred))
        
        # Correlation
        mask = ~(np.isnan(actual) | np.isnan(pred))
        if mask.sum() > 10:
            corr = np.corrcoef(actual[mask], pred[mask])[0,1]
        else:
            corr = 0.0
        
        # MAE and RMSE
        mae = np.mean(np.abs(actual - pred))
        rmse = np.sqrt(np.mean((actual - pred)**2))
        
        # Sharpe-like metric (return/volatility of predictions)
        pred_sharpe = np.mean(pred) / (np.std(pred) + 1e-8)
        
        metrics[f'Day_{horizon}'] = {
            'direction_accuracy': direction_acc,
            'correlation': corr,
            'mae': mae,
            'rmse': rmse,
            'pred_sharpe': pred_sharpe
        }
        
        print(f"Day {horizon:3d} | Dir: {direction_acc:.2%} | Corr: {corr:+.3f} | "
              f"MAE: {mae:.2f}% | RMSE: {rmse:.2f}%")
    
    print("="*80 + "\n")
    
    return metrics

# ==============================================================================
# MAIN TRAINING PIPELINE
# ==============================================================================

def train_integrated_system():
    """Main training pipeline with all enhancements"""
    
    print("\n" + "="*80)
    print(" NIFTY-50 INTEGRATED TRAINING SYSTEM v2.0")
    print("="*80)
    print(f"\nConfiguration:")
    print(f"  Ensemble: {'✓ Enabled' if CONFIG['ENABLE_ENSEMBLE'] else '✗ Disabled'}")
    print(f"  Market Features: {'✓ Enabled' if CONFIG['ENABLE_MARKET_FEATURES'] else '✗ Disabled'}")
    print(f"  Custom Loss: {'✓ Enabled' if CONFIG['ENABLE_CUSTOM_LOSS'] else '✗ Disabled'}")
    print(f"  N Models: {CONFIG['N_ENSEMBLE']}")
    print(f"  Sequence Length: {CONFIG['SEQ_LENGTH']}")
    print(f"  Max Horizon: {CONFIG['MAX_HORIZON']} days")
    print("="*80)
    
    # Phase 1: Download NIFTY for market context
    print("\n=== PHASE 1: Downloading Market Data ===")
    
    nifty_df = None
    if CONFIG['ENABLE_MARKET_FEATURES']:
        print("Downloading NIFTY index...")
        nifty_df = yf.download('^NSEI', period=f"{CONFIG['TRAIN_YEARS']}y", interval='1d', progress=False)
        if isinstance(nifty_df.columns, pd.MultiIndex):
            nifty_df.columns = nifty_df.columns.get_level_values(0)
        print(f"✓ NIFTY data: {len(nifty_df)} days")
    else:
        print("Market features disabled, skipping NIFTY download")
    
    # Phase 2: Process all stocks
    print("\n=== PHASE 2: Processing Stocks ===")
    
    stock_data = []
    
    for ticker in CONFIG['TICKERS']:
        df = prepare_stock_features(ticker, nifty_df)
        if df is not None:
            stock_data.append((ticker, df))
    
    print(f"\n✅ Successfully processed {len(stock_data)}/{len(CONFIG['TICKERS'])} stocks")
    
    if len(stock_data) == 0:
        print("❌ No valid data. Exiting.")
        return
    
    # Phase 3: Create features and targets
    print("\n=== PHASE 3: Creating Sequences ===")
    
    feature_cols = get_feature_columns()
    print(f"Using {len(feature_cols)} features: {feature_cols}")
    
    all_X = []
    all_y = []
    
    for ticker, df in stock_data:
        # Calculate targets
        targets_df = calculate_targets(df)
        df = pd.concat([df, targets_df], axis=1)
        df.dropna(inplace=True)
        
        # Extract features and targets
        features = df[feature_cols].values
        targets = df[[f'Target_Day_{i}' for i in range(1, CONFIG['MAX_HORIZON']+1)]].values
        
        # Create sequences
        X_stock, y_stock = create_sequences_safe(features, targets, CONFIG['SEQ_LENGTH'])
        
        if len(X_stock) > 0:
            all_X.append(X_stock)
            all_y.append(y_stock)
            print(f"  {ticker:15s}: {len(X_stock):5,} sequences")
    
    # Concatenate all
    X_data = np.concatenate(all_X)
    y_data = np.concatenate(all_y)
    
    print(f"\n✅ Total sequences: {len(X_data):,}")
    print(f"   Shape: X={X_data.shape}, y={y_data.shape}")
    
    # Phase 4: Fit scaler and transform
    print("\n=== PHASE 4: Fitting Scaler ===")
    
    # Reshape for scaling
    n_samples, seq_len, n_features = X_data.shape
    X_reshaped = X_data.reshape(-1, n_features)
    
    # Fit on 80% to avoid leakage
    split_idx = int(len(X_reshaped) * 0.8)
    scaler = StandardScaler()
    scaler.fit(X_reshaped[:split_idx])
    
    # Transform all data
    X_scaled = scaler.transform(X_reshaped)
    X_scaled = X_scaled.reshape(n_samples, seq_len, n_features)
    
    # Save scaler
    joblib.dump(scaler, CONFIG['SCALER_NAME'])
    print(f"✅ Scaler fitted and saved to {CONFIG['SCALER_NAME']}")
    
    # Phase 5: Train/Val Split
    print("\n=== PHASE 5: Creating Train/Val Split ===")
    
    split_idx = int(len(X_scaled) * 0.8)
    gap = CONFIG['MAX_HORIZON']
    
    X_train = X_scaled[:split_idx]
    y_train = y_data[:split_idx]
    X_val = X_scaled[split_idx + gap:]
    y_val = y_data[split_idx + gap:]
    
    print(f"Train: {len(X_train):,} samples")
    print(f"Val:   {len(X_val):,} samples")
    print(f"Gap:   {gap} days")
    
    # Phase 6: Train model(s)
    print("\n=== PHASE 6: Training Model(s) ===")
    
    if CONFIG['ENABLE_ENSEMBLE']:
        # Train ensemble
        ensemble = ModelEnsemble(n_models=CONFIG['N_ENSEMBLE'])
        ensemble.train(X_train, y_train, X_val, y_val, 
                      CONFIG['SEQ_LENGTH'], n_features, CONFIG['MAX_HORIZON'])
        predictor = ensemble
        
        # Save ensemble info
        ensemble_info = {
            'n_models': CONFIG['N_ENSEMBLE'],
            'weights': ensemble.weights.tolist(),
            'model_dir': CONFIG['ENSEMBLE_DIR']
        }
        joblib.dump(ensemble_info, f"{CONFIG['MODEL_PREFIX']}_ensemble_info.pkl")
        
    else:
        # Train single model
        print("Training single model...")
        model = build_enhanced_model(CONFIG['SEQ_LENGTH'], n_features, CONFIG['MAX_HORIZON'])
        
        if CONFIG['ENABLE_CUSTOM_LOSS']:
            loss = DirectionAwareLoss(direction_weight=0.3)
        else:
            loss = 'mse'
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss=loss,
            metrics=['mae']
        )
        
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
            ModelCheckpoint(f"{CONFIG['MODEL_PREFIX']}_single.keras", save_best_only=True, monitor='val_loss', verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
        ]
        
        model.fit(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=CONFIG['EPOCHS'], batch_size=CONFIG['BATCH_SIZE'],
                 callbacks=callbacks, verbose=1)
        
        predictor = model
    
    # Phase 7: Evaluation
    print("\n=== PHASE 7: Model Evaluation ===")
    
    metrics = evaluate_model(predictor, X_val, y_val)
    
    # Save metrics
    metrics_data = {
        'config': CONFIG,
        'horizon_metrics': metrics,
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'n_features': n_features
    }
    joblib.dump(metrics_data, CONFIG['METRICS_NAME'])
    
    # Phase 8: Summary
    print("\n" + "="*80)
    print(" TRAINING COMPLETE!")
    print("="*80)
    print(f"\n📁 Generated Files:")
    print(f"   • Scaler: {CONFIG['SCALER_NAME']}")
    print(f"   • Metrics: {CONFIG['METRICS_NAME']}")
    
    if CONFIG['ENABLE_ENSEMBLE']:
        print(f"   • Ensemble: {CONFIG['ENSEMBLE_DIR']}/ ({CONFIG['N_ENSEMBLE']} models)")
        print(f"   • Ensemble Info: {CONFIG['MODEL_PREFIX']}_ensemble_info.pkl")
    else:
        print(f"   • Model: {CONFIG['MODEL_PREFIX']}_single.keras")
    
    print(f"\n📊 Key Metrics:")
    for horizon in [20, 60, 120, 180]:
        if f'Day_{horizon}' in metrics:
            m = metrics[f'Day_{horizon}']
            print(f"   Day {horizon:3d}: Dir={m['direction_accuracy']:.1%}, "
                  f"Corr={m['correlation']:+.3f}, MAE={m['mae']:.2f}%")
    
    print("\n" + "="*80)
    print("Ready for inference! Use the enhanced scoring script.")
    print("="*80 + "\n")

# ==============================================================================
# ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    train_integrated_system()


 NIFTY-50 INTEGRATED TRAINING SYSTEM v2.0

Configuration:
  Ensemble: ✓ Enabled
  Market Features: ✓ Enabled
  Custom Loss: ✓ Enabled
  N Models: 5
  Sequence Length: 60
  Max Horizon: 180 days

=== PHASE 1: Downloading Market Data ===
✓ NIFTY data: 4465 days

=== PHASE 2: Processing Stocks ===
  → Processing RELIANCE.NS...
    ✓ 4429 samples
  → Processing TCS.NS...
    ✓ 4429 samples
  → Processing HDFCBANK.NS...
    ✓ 4429 samples
  → Processing INFY.NS...
    ✓ 4429 samples
  → Processing ICICIBANK.NS...
    ✓ 4429 samples
  → Processing HINDUNILVR.NS...
    ✓ 4429 samples
  → Processing SBIN.NS...
    ✓ 4429 samples
  → Processing BHARTIARTL.NS...
    ✓ 4429 samples
  → Processing ITC.NS...
    ✓ 4429 samples
  → Processing KOTAKBANK.NS...
    ✓ 4429 samples
  → Processing LICI.NS...
    ✓ 817 samples
  → Processing LT.NS...
    ✓ 4429 samples
  → Processing AXISBANK.NS...
    ✓ 4429 samples
  → Processing HCLTECH.NS...
    ✓ 4430 samples
  → Processing ASIANPAINT.NS...
    ✓ 442

# NIFTY-50 Stock Predictor - Inference System v2.0## 📋 OverviewThis cell implements the **inference pipeline** for making predictions on new stocks using the trained ensemble model. It handles feature calculation, model loading, and generates technical scores with uncertainty estimates.---## 🔧 Key Components### 1. **Model Loading**- Auto-detects ensemble vs single model- Loads custom loss function (`DirectionAwareLoss`)- Handles 5 ensemble models with performance-based weights### 2. **Feature Preparation**- Downloads latest 2 years of stock data- Calculates same 15 features as training- Ensures feature alignment with NIFTY index### 3. **Prediction Engine**- Scales features using saved scaler- Generates predictions for all 180 horizons- Returns mean prediction + uncertainty (std across ensemble)### 4. **Scoring System**Converts raw predictions into actionable scores (0-100):**Score = 50 + (Z-Score × Confidence)**Where:- **Z-Score**: Prediction vs market baseline, normalized by uncertainty- **Confidence**: Adjusted based on ADX, RSI, ensemble disagreement**Signal Thresholds**:- 75+ → STRONG BUY 🟢🟢- 65-75 → BUY 🟢- 55-65 → WEAK BUY- 45-55 → NEUTRAL ⚪- 35-45 → WEAK SELL- 25-35 → SELL 🔴- <25 → STRONG SELL 🔴🔴---## 📊 Usage Examples### Single Stock Analysis```python# Analyze a single stockresult = analyze_technical_score('RELIANCE.NS', duration_days=30)print_analysis_report(result)```### Portfolio Screening```python# Analyze multiple stockstickers = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS']results = analyze_portfolio(tickers, duration_days=20)```---## 🎯 Output Interpretation### Technical Score- **0-100 scale**: Higher = more bullish- **50 = neutral**: Expected to match market baseline- **Confidence multiplier**: Adjusts score based on market conditions### Uncertainty- **Low (<2%)**: High ensemble agreement, strong signal- **Medium (2-5%)**: Moderate agreement- **High (>5%)**: Low agreement, use caution### IndicatorsCurrent values of all 15 features for context and validation---## ⚠️ Important Notes- **Requires trained models**: Run training cell first- **Internet required**: Downloads latest stock data via yfinance- **Market hours**: Data may be delayed 15-20 minutes- **Ensemble advantage**: Provides uncertainty estimates for risk management

In [11]:
"""
NIFTY-50 Multi-Horizon Stock Predictor - Inference System v2.0 (FIXED)

Fixes:
- Custom loss function registration
- Proper model loading with custom objects
- Better error handling

Author: Enhanced Inference Pipeline
Date: 2025
"""

import os
import glob
import joblib
import yfinance as yf
import pandas as pd
import numpy as np
import tensorflow as tf
import warnings
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

warnings.filterwarnings("ignore")

# ==========================================
# CUSTOM LOSS (Must be defined before loading models)
# ==========================================
class DirectionAwareLoss(tf.keras.losses.Loss):
    """Penalize wrong direction predictions more - Compatible with TF 2.x"""
    def __init__(self, direction_weight=0.3, name='direction_aware_loss', reduction='sum_over_batch_size'):
        super().__init__(name=name, reduction=reduction)
        self.direction_weight = direction_weight
        
    def call(self, y_true, y_pred):
        # MSE component
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        
        # Direction penalty
        direction_penalty = tf.reduce_mean(
            tf.cast(tf.sign(y_true) != tf.sign(y_pred), tf.float32)
        )
        
        return (1 - self.direction_weight) * mse + self.direction_weight * direction_penalty * 100
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'direction_weight': self.direction_weight
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)

# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    'SEQ_LENGTH': 60,
    'MAX_HORIZON': 180,
    'SCALER_PATH': 'scaler_v2.pkl',
    'ENSEMBLE_DIR': 'ensemble_models',
    'ENSEMBLE_INFO_PATH': 'nifty50_integrated_ensemble_info.pkl',
    'SINGLE_MODEL_PATH': 'nifty50_integrated_single.keras',
    'USE_ENSEMBLE': True,  # Auto-detect if ensemble exists
}

# ==========================================
# INDICATORS (Must match training)
# ==========================================
class EnhancedIndicators:
    @staticmethod
    def get_rsi(series, period=14):
        delta = series.diff()
        gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
        loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))

    @staticmethod
    def get_adx(high, low, close, period=14):
        plus_dm = high.diff()
        minus_dm = low.diff()
        plus_dm = plus_dm.where(plus_dm > 0, 0)
        minus_dm = minus_dm.where(minus_dm < 0, 0).abs()
        
        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low - close.shift(1)).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        
        atr = tr.ewm(alpha=1/period, adjust=False).mean()
        plus_di = 100 * (plus_dm.ewm(alpha=1/period, adjust=False).mean() / atr)
        minus_di = 100 * (minus_dm.ewm(alpha=1/period, adjust=False).mean() / atr)
        dx = (abs(plus_di - minus_di) / (plus_di + minus_di + 1e-8)) * 100
        return dx.ewm(alpha=1/period, adjust=False).mean()

    @staticmethod
    def get_natr(high, low, close, period=14):
        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low - close.shift(1)).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1/period, adjust=False).mean()
        return (atr / close) * 100

    @staticmethod
    def get_obv_slope(close, volume, period=14):
        obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
        return obv.diff(period)

    @staticmethod
    def get_dist_sma(close, period=50):
        sma = close.rolling(period).mean()
        return (close - sma) / sma
    
    @staticmethod
    def get_macd(close, fast=12, slow=26, signal=9):
        ema_fast = close.ewm(span=fast, adjust=False).mean()
        ema_slow = close.ewm(span=slow, adjust=False).mean()
        macd = ema_fast - ema_slow
        macd_signal = macd.ewm(span=signal, adjust=False).mean()
        return macd - macd_signal
    
    @staticmethod
    def get_roc(close, period=10):
        return ((close - close.shift(period)) / close.shift(period)) * 100
    
    @staticmethod
    def get_volume_ratio(volume, period=20):
        vol_ma = volume.rolling(period).mean()
        return volume / vol_ma
    
    @staticmethod
    def get_bollinger_position(close, period=20, std=2):
        sma = close.rolling(period).mean()
        rolling_std = close.rolling(period).std()
        upper = sma + (rolling_std * std)
        lower = sma - (rolling_std * std)
        # Avoid division by zero
        band_width = upper - lower
        band_width = band_width.replace(0, np.nan)
        return (close - lower) / band_width

# ==========================================
# MARKET CONTEXT
# ==========================================
class MarketContextFeatures:
    @staticmethod
    def calculate_market_features(nifty_close):
        features = pd.DataFrame(index=nifty_close.index)
        
        for period in [20, 50, 200]:
            ma = nifty_close.rolling(period).mean()
            features[f'above_ma_{period}'] = (nifty_close > ma).astype(float)
        
        features['market_strength'] = features[[f'above_ma_{p}' for p in [20, 50, 200]]].mean(axis=1)
        features['market_momentum'] = nifty_close.pct_change(20) * 100
        
        returns = nifty_close.pct_change()
        features['market_volatility'] = returns.rolling(20).std() * np.sqrt(252) * 100
        
        # Fix division by zero
        rolling_std = returns.rolling(20).std()
        features['market_trend'] = returns.rolling(20).mean() / (rolling_std + 1e-8)
        
        return features[['market_strength', 'market_momentum', 'market_volatility', 'market_trend']]

# ==========================================
# MODEL LOADER
# ==========================================
class ModelLoader:
    """Load either ensemble or single model with proper custom object handling"""
    
    def __init__(self):
        self.scaler = None
        self.models = []
        self.weights = []
        self.is_ensemble = False
        self._load_models()
    
    def _get_custom_objects(self):
        """Return dictionary of custom objects for model loading"""
        return {
            'DirectionAwareLoss': DirectionAwareLoss
        }
    
    def _load_models(self):
        """Auto-detect and load appropriate model type"""
        
        # Load scaler
        if not os.path.exists(CONFIG['SCALER_PATH']):
            raise FileNotFoundError(f"Scaler not found: {CONFIG['SCALER_PATH']}")
        
        self.scaler = joblib.load(CONFIG['SCALER_PATH'])
        print(f"✓ Loaded scaler: {CONFIG['SCALER_PATH']}")
        
        # Try to load ensemble first
        if os.path.exists(CONFIG['ENSEMBLE_DIR']) and os.path.exists(CONFIG['ENSEMBLE_INFO_PATH']):
            print(f"✓ Detected ensemble in {CONFIG['ENSEMBLE_DIR']}")
            self._load_ensemble()
            self.is_ensemble = True
        
        # Fallback to single model
        elif os.path.exists(CONFIG['SINGLE_MODEL_PATH']):
            print(f"✓ Loading single model: {CONFIG['SINGLE_MODEL_PATH']}")
            model = load_model(CONFIG['SINGLE_MODEL_PATH'], 
                             custom_objects=self._get_custom_objects(),
                             compile=False)
            self.models = [model]
            self.weights = [1.0]
            self.is_ensemble = False
        
        else:
            raise FileNotFoundError("No trained model found. Please run training first.")
    
    def _load_ensemble(self):
        """Load all ensemble models with custom objects"""
        ensemble_info = joblib.load(CONFIG['ENSEMBLE_INFO_PATH'])
        self.weights = np.array(ensemble_info['weights'])
        
        model_files = sorted(glob.glob(os.path.join(CONFIG['ENSEMBLE_DIR'], 'model_*.keras')))
        
        print(f"Loading {len(model_files)} ensemble models...")
        for i, model_file in enumerate(model_files, 1):
            try:
                # Try with custom objects first
                try:
                    model = load_model(model_file, 
                                     custom_objects=self._get_custom_objects(),
                                     compile=False)
                except:
                    # If that fails, try loading with safe_mode
                    print(f"  ⚠ Trying safe_mode for model {i}...")
                    model = tf.keras.models.load_model(model_file, compile=False)
                
                self.models.append(model)
                print(f"  ✓ Model {i}/{len(model_files)}")
            except Exception as e:
                print(f"  ✗ Failed to load {model_file}: {e}")
                raise
        
        print(f"✓ Successfully loaded {len(self.models)} ensemble models")
        print(f"  Weights: {[f'{w:.3f}' for w in self.weights]}")
    
    def predict(self, X):
        """Predict with loaded model(s)"""
        if self.is_ensemble:
            predictions = []
            for model in self.models:
                pred = model.predict(X, verbose=0)
                predictions.append(pred)
            
            predictions = np.array(predictions)
            weighted_pred = np.average(predictions, axis=0, weights=self.weights)
            std_pred = np.std(predictions, axis=0)
            
            return weighted_pred, std_pred
        else:
            pred = self.models[0].predict(X, verbose=0)
            return pred, np.zeros_like(pred)  # No uncertainty for single model

# ==========================================
# MARKET BASELINE
# ==========================================
class MarketBaseline:
    @staticmethod
    def get_nifty_baseline():
        try:
            nifty = yf.download('^NSEI', period='10y', interval='1d', progress=False)
            if len(nifty) < 252:
                return {'annual_return': 12.0, 'annual_vol': 15.0}
            
            returns = nifty['Close'].pct_change().dropna()
            annual_return = (nifty['Close'].pct_change(252).mean() * 100)
            annual_vol = (returns.std() * np.sqrt(252) * 100)
            
            return {'annual_return': annual_return, 'annual_vol': annual_vol}
        except:
            return {'annual_return': 12.0, 'annual_vol': 15.0}

# ==========================================
# FEATURE PREPARATION
# ==========================================
def prepare_features_for_inference(ticker, nifty_df=None):
    """Prepare features matching training pipeline"""
    
    # Download stock data
    df = yf.download(ticker, period="2y", interval="1d", progress=False)
    
    if len(df) < 150:
        raise ValueError("Insufficient historical data")
    
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Calculate stock indicators
    df['RSI'] = EnhancedIndicators.get_rsi(df['Close'])
    df['ADX'] = EnhancedIndicators.get_adx(df['High'], df['Low'], df['Close'])
    df['NATR'] = EnhancedIndicators.get_natr(df['High'], df['Low'], df['Close'])
    df['OBV_Slope'] = EnhancedIndicators.get_obv_slope(df['Close'], df['Volume'])
    df['Dist_SMA'] = EnhancedIndicators.get_dist_sma(df['Close'])
    df['MACD'] = EnhancedIndicators.get_macd(df['Close'])
    df['ROC'] = EnhancedIndicators.get_roc(df['Close'])
    df['Vol_Ratio'] = EnhancedIndicators.get_volume_ratio(df['Volume'])
    df['BB_Position'] = EnhancedIndicators.get_bollinger_position(df['Close'])
    
    # Add market context if NIFTY data provided
    if nifty_df is not None:
        nifty_aligned = nifty_df.reindex(df.index, method='ffill')
        
        market_features = MarketContextFeatures.calculate_market_features(nifty_aligned['Close'])
        df = pd.concat([df, market_features], axis=1)
        
        stock_returns = df['Close'].pct_change(20)
        market_returns = nifty_aligned['Close'].pct_change(20)
        df['relative_strength'] = stock_returns - market_returns
        
        stock_ret = df['Close'].pct_change()
        market_ret = nifty_aligned['Close'].pct_change()
        df['beta'] = (
            stock_ret.rolling(60).cov(market_ret) / 
            (market_ret.rolling(60).var() + 1e-8)
        )
    
    df.dropna(inplace=True)
    
    return df

# ==========================================
# SCORING ENGINE
# ==========================================
def analyze_technical_score(ticker, duration_days, model_loader=None, verbose=True):
    """
    Complete technical analysis with ensemble support
    """
    
    result = {
        'ticker': ticker,
        'duration': duration_days,
        'technical_score': 50,
        'predicted_return': 0.0,
        'uncertainty': 0.0,
        'confidence': 0.0,
        'signal': 'NEUTRAL',
        'status': 'FAIL',
        'message': '',
        'indicators': {},
        'is_ensemble': False
    }
    
    try:
        # Load models if not provided
        if model_loader is None:
            model_loader = ModelLoader()
        
        result['is_ensemble'] = model_loader.is_ensemble
        
        # Download NIFTY for market context
        try:
            nifty_df = yf.download('^NSEI', period='2y', interval='1d', progress=False)
            if isinstance(nifty_df.columns, pd.MultiIndex):
                nifty_df.columns = nifty_df.columns.get_level_values(0)
        except Exception as e:
            result['message'] = f"Failed to download NIFTY data: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        # Prepare features
        try:
            df = prepare_features_for_inference(ticker, nifty_df)
        except Exception as e:
            result['message'] = f"Feature preparation failed: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        if len(df) < CONFIG['SEQ_LENGTH']:
            result['message'] = "Insufficient sequence length"
            return result
        
        # Store current indicators
        try:
            result['indicators'] = {
                'RSI': round(float(df['RSI'].iloc[-1]), 2),
                'ADX': round(float(df['ADX'].iloc[-1]), 2),
                'NATR': round(float(df['NATR'].iloc[-1]), 2),
                'MACD': round(float(df['MACD'].iloc[-1]), 4),
                'ROC': round(float(df['ROC'].iloc[-1]), 2),
                'BB_Position': round(float(df['BB_Position'].iloc[-1]), 2)
            }
            
            if 'market_strength' in df.columns:
                result['indicators']['Market_Strength'] = round(float(df['market_strength'].iloc[-1]), 2)
                result['indicators']['Beta'] = round(float(df['beta'].iloc[-1]), 2)
        except Exception as e:
            result['message'] = f"Indicator extraction failed: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        # Prepare input sequence
        feature_cols = [
            'RSI', 'ADX', 'NATR', 'OBV_Slope', 'Dist_SMA',
            'MACD', 'ROC', 'Vol_Ratio', 'BB_Position'
        ]
        
        if 'market_strength' in df.columns:
            feature_cols.extend([
                'market_strength', 'market_momentum', 'market_volatility', 'market_trend',
                'relative_strength', 'beta'
            ])
        
        try:
            last_seq = df[feature_cols].tail(CONFIG['SEQ_LENGTH'])
            
            # Scale features
            input_seq = model_loader.scaler.transform(last_seq)
            input_seq = np.array([input_seq])
        except Exception as e:
            result['message'] = f"Sequence preparation failed: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        # Predict
        try:
            mean_forecast, std_forecast = model_loader.predict(input_seq)
            mean_forecast = mean_forecast[0]
            std_forecast = std_forecast[0]
        except Exception as e:
            result['message'] = f"Prediction failed: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        # Get prediction for requested horizon
        eff_duration = min(duration_days, CONFIG['MAX_HORIZON'])
        raw_return = float(mean_forecast[eff_duration - 1])
        prediction_std = float(std_forecast[eff_duration - 1])
        
        # === SCORING LOGIC ===
        
        # Market baseline
        market_baseline = MarketBaseline.get_nifty_baseline()
        time_factor = eff_duration / 365.0
        period_baseline_return = float(market_baseline['annual_return'] * time_factor)
        period_baseline_vol = float(market_baseline['annual_vol'] * np.sqrt(time_factor))
        
        # Stock volatility
        current_natr = float(df['NATR'].iloc[-1])
        expected_vol_move = float(current_natr * np.sqrt(eff_duration))
        
        # Calculate Z-score
        total_uncertainty = float(np.sqrt(expected_vol_move**2 + period_baseline_vol**2 + prediction_std**2))
        
        if total_uncertainty > 0:
            final_z = float((raw_return - period_baseline_return) / total_uncertainty)
        else:
            final_z = 0.0
        
        # Confidence adjustment
        current_adx = float(df['ADX'].iloc[-1])
        current_rsi = float(df['RSI'].iloc[-1])
        
        confidence = 1.0
        
        if current_adx < 20:
            confidence *= 0.7
        elif current_adx > 40:
            confidence *= 1.1
        
        if current_rsi > 70 and raw_return > 0:
            confidence *= 0.8
        elif current_rsi < 30 and raw_return < 0:
            confidence *= 0.8
        
        if prediction_std > 5:
            confidence *= 0.85
        
        # Ensemble bonus
        if model_loader.is_ensemble:
            confidence *= 1.05  # Slight boost for ensemble
        
        confidence = min(confidence, 1.2)
        
        # Convert to score
        base_score = norm.cdf(final_z) * 100
        final_score = 50 + (base_score - 50) * confidence
        final_score = np.clip(final_score, 0, 100)
        
        # Determine signal
        if final_score > 75:
            signal = 'STRONG BUY'
        elif final_score > 65:
            signal = 'BUY'
        elif final_score > 55:
            signal = 'WEAK BUY'
        elif final_score > 45:
            signal = 'NEUTRAL'
        elif final_score > 35:
            signal = 'WEAK SELL'
        elif final_score > 25:
            signal = 'SELL'
        else:
            signal = 'STRONG SELL'
        
        # Update result
        result.update({
            'technical_score': round(float(final_score), 2),
            'predicted_return': round(float(raw_return), 2),
            'uncertainty': round(float(prediction_std), 2),
            'confidence': round(float(confidence), 2),
            'signal': signal,
            'status': 'SUCCESS',
            'z_score': round(float(final_z), 3),
            'market_baseline': round(float(period_baseline_return), 2)
        })
    
    except Exception as e:
        import traceback
        result['message'] = f"Error: {str(e)}"
        if verbose:
            print(f"❌ {result['message']}")
            print(traceback.format_exc())
    
    return result

# ==========================================
# REPORTING
# ==========================================
def print_analysis_report(result):
    """Enhanced report with ensemble info"""
    
    print("\n" + "="*80)
    print(f" TECHNICAL ANALYSIS REPORT: {result['ticker']}")
    print("="*80)
    
    if result['status'] != 'SUCCESS':
        print(f"\n❌ Analysis Failed: {result['message']}")
        print("="*80)
        return
    
    # Model info
    model_type = "🎯 ENSEMBLE" if result['is_ensemble'] else "📊 SINGLE MODEL"
    print(f"\n{model_type}")
    
    print(f"\n📊 PREDICTION ({result['duration']}-Day Horizon)")
    print(f"   Expected Return:    {result['predicted_return']:+.2f}%")
    print(f"   Uncertainty (±):    {result['uncertainty']:.2f}%")
    print(f"   Market Baseline:    {result['market_baseline']:+.2f}%")
    print(f"   Z-Score:           {result['z_score']:+.3f}")
    
    print(f"\n🎯 SCORING")
    print(f"   Technical Score:    {result['technical_score']:.1f} / 100")
    print(f"   Model Confidence:   {result['confidence']:.2f}x")
    
    score = result['technical_score']
    if score > 75:
        emoji = "🟢🟢"
    elif score > 65:
        emoji = "🟢"
    elif score < 25:
        emoji = "🔴🔴"
    elif score < 35:
        emoji = "🔴"
    else:
        emoji = "⚪"
    
    print(f"   Signal:             {emoji} {result['signal']}")
    
    print(f"\n📈 CURRENT INDICATORS")
    for name, value in result['indicators'].items():
        print(f"   {name:18s}: {value}")
    
    print("\n" + "="*80 + "\n")

# ==========================================
# BATCH ANALYSIS
# ==========================================
def analyze_portfolio(tickers, duration_days, model_loader=None):
    """Analyze multiple stocks efficiently"""
    
    if model_loader is None:
        model_loader = ModelLoader()
    
    print(f"\n{'='*80}")
    print(f" PORTFOLIO ANALYSIS - {duration_days} Day Horizon")
    print(f" Model: {'Ensemble' if model_loader.is_ensemble else 'Single'}")
    print(f"{'='*80}\n")
    
    results = []
    
    for ticker in tickers:
        print(f"Analyzing {ticker:15s}...", end=' ')
        result = analyze_technical_score(ticker, duration_days, model_loader, verbose=False)
        
        if result['status'] == 'SUCCESS':
            print(f"✓ Score: {result['technical_score']:.1f}")
            results.append(result)
        else:
            print(f"✗ Failed")
    
    # Sort by score
    results.sort(key=lambda x: x['technical_score'], reverse=True)
    
    # Print ranking
    print(f"\n{'='*80}")
    print(f" RANKING BY TECHNICAL SCORE")
    print(f"{'='*80}")
    print(f"{'Rank':<6} {'Ticker':<15} {'Score':<10} {'Return':<12} {'Signal':<15}")
    print("-"*80)
    
    for i, res in enumerate(results, 1):
        print(f"{i:<6} {res['ticker']:<15} {res['technical_score']:<10.1f} "
              f"{res['predicted_return']:>+6.2f}%     {res['signal']:<15}")
    
    print("="*80 + "\n")
    
    return results

# ==========================================
# MAIN
# ==========================================
if __name__ == "__main__":
    
    print("\n" + "="*80)
    print(" INTEGRATED INFERENCE SYSTEM v2.0")
    print("="*80)
    
    # Initialize model loader once
    try:
        model_loader = ModelLoader()
    except Exception as e:
        print(f"\n❌ Failed to load models: {e}")
        print("\nPlease run the training script first:")
        print("  python integrated_training_v2.py")
        exit(1)
    
    # Single stock analysis
    print("\n" + "="*80)
    print(" SINGLE STOCK ANALYSIS")
    print("="*80)
    
    test_ticker = "RELIANCE.NS"
    test_duration = 30
    
    result = analyze_technical_score(test_ticker, test_duration, model_loader)
    print_analysis_report(result)
    
    # Portfolio analysis
    print("\n" + "="*80)
    print(" PORTFOLIO SCREENING")
    print("="*80)
    
    portfolio = [
        'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS',
        'ICICIBANK.NS', 'HINDUNILVR.NS', 'SBIN.NS', 'ITC.NS'
    ]
    
    portfolio_results = analyze_portfolio(portfolio, 20, model_loader)
    
    # Top picks
    print("\n🏆 TOP 3 PICKS:")
    for i, res in enumerate(portfolio_results[:3], 1):
        print(f"   {i}. {res['ticker']:15s} - Score: {res['technical_score']:.1f}, "
              f"Return: {res['predicted_return']:+.2f}%")


 INTEGRATED INFERENCE SYSTEM v2.0
✓ Loaded scaler: scaler_v2.pkl
✓ Detected ensemble in ensemble_models
Loading 5 ensemble models...
  ✓ Model 1/5
  ✓ Model 2/5
  ✓ Model 3/5
  ✓ Model 4/5
  ✓ Model 5/5
✓ Successfully loaded 5 ensemble models
  Weights: ['0.207', '0.208', '0.195', '0.195', '0.195']

 SINGLE STOCK ANALYSIS

 TECHNICAL ANALYSIS REPORT: RELIANCE.NS

🎯 ENSEMBLE

📊 PREDICTION (30-Day Horizon)
   Expected Return:    +2.04%
   Uncertainty (±):    0.84%
   Market Baseline:    +1.19%
   Z-Score:           +0.096

🎯 SCORING
   Technical Score:    53.5 / 100
   Model Confidence:   0.92x
   Signal:             ⚪ NEUTRAL

📈 CURRENT INDICATORS
   RSI               : 72.96
   ADX               : 50.79
   NATR              : 1.35
   MACD              : 1.9933
   ROC               : 3.2
   BB_Position       : 0.9
   Market_Strength   : 1.0
   Beta              : 1.12



 PORTFOLIO SCREENING

 PORTFOLIO ANALYSIS - 20 Day Horizon
 Model: Ensemble

Analyzing RELIANCE.NS    ... ✓ Score: 5

In [15]:
"""
Comprehensive Backtesting System for NIFTY-50 Prediction Model

This script:
1. Validates if technical scores predict future returns
2. Tests different trading strategies based on scores
3. Calculates realistic P&L with transaction costs
4. Compares to buy-and-hold and market benchmarks
5. Provides detailed performance metrics

Author: Backtesting & Validation Pipeline
Date: 2025
"""

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy.stats import spearmanr, pearsonr
import warnings
warnings.filterwarnings('ignore')

# NOTE: This assumes ModelLoader, CONFIG, EnhancedIndicators, and MarketContextFeatures
# are already defined in previous cells of your Jupyter notebook

# ==========================================
# CONFIGURATION
# ==========================================
BACKTEST_CONFIG = {
    'TEST_START_DATE': '2024-01-01',  # Start of out-of-sample period
    'INITIAL_CAPITAL': 1000000,  # 10 lakhs
    'TRANSACTION_COST': 0.002,  # 0.2% per trade (realistic for India)
    'MAX_POSITION_SIZE': 0.15,  # Max 15% per stock
    'REBALANCE_FREQUENCY': 20,  # Days between rebalancing
    'STOP_LOSS': 0.08,  # 8% stop loss
    'TAKE_PROFIT': 0.20,  # 20% take profit
}

# ==========================================
# BACKTESTING ENGINE
# ==========================================
class ModelBacktester:
    """
    Walk-forward backtesting system that simulates real trading
    """
    
    def __init__(self, model_loader, config=BACKTEST_CONFIG):
        self.model_loader = model_loader
        self.config = config
        self.results = []
        
    def backtest_single_stock(self, ticker, horizon=60, verbose=True):
        """
        Backtest a single stock with walk-forward validation
        
        Returns:
            DataFrame with dates, predictions, actuals, scores
        """
        
        if verbose:
            print(f"\n{'='*80}")
            print(f" BACKTESTING: {ticker} | Horizon: {horizon} days")
            print(f"{'='*80}")
        
        try:
            # Download full history
            df = yf.download(ticker, start='2020-01-01', interval='1d', progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            
            # Download NIFTY
            nifty = yf.download('^NSEI', start='2020-01-01', interval='1d', progress=False)
            if isinstance(nifty.columns, pd.MultiIndex):
                nifty.columns = nifty.columns.get_level_values(0)
            
            # Get test period
            test_start = pd.to_datetime(self.config['TEST_START_DATE'])
            test_data = df[df.index >= test_start].copy()
            
            if len(test_data) < horizon + 60:
                if verbose:
                    print(f"  ⚠️  Insufficient test data")
                return None
            
            # Prepare features for full dataset
            df_features = self._prepare_features(df, nifty)
            
            backtest_results = []
            
            # Make predictions at regular intervals
            prediction_dates = test_data.index[::self.config['REBALANCE_FREQUENCY']]
            
            for i, pred_date in enumerate(prediction_dates):
                # Skip if not enough future data
                future_date = pred_date + timedelta(days=horizon)
                if future_date not in df.index:
                    continue
                
                # Get historical data up to prediction date
                hist_data = df_features[df_features.index <= pred_date]
                
                if len(hist_data) < CONFIG['SEQ_LENGTH']:
                    continue
                
                try:
                    # Make prediction
                    pred_return, score, uncertainty = self._predict_at_date(
                        hist_data, horizon, ticker, pred_date
                    )
                    
                    # Get actual return
                    actual_return = self._get_actual_return(df, pred_date, horizon)
                    
                    if actual_return is not None:
                        # Record result
                        backtest_results.append({
                            'date': pred_date,
                            'ticker': ticker,
                            'horizon': horizon,
                            'predicted_return': pred_return,
                            'actual_return': actual_return,
                            'technical_score': score,
                            'uncertainty': uncertainty,
                            'pred_direction': np.sign(pred_return),
                            'actual_direction': np.sign(actual_return),
                            'direction_correct': np.sign(pred_return) == np.sign(actual_return)
                        })
                
                except Exception as e:
                    if verbose:
                        print(f"  ⚠️  Error at {pred_date}: {e}")
                    continue
            
            if not backtest_results:
                if verbose:
                    print(f"  ❌ No valid predictions")
                return None
            
            # Convert to DataFrame
            results_df = pd.DataFrame(backtest_results)
            
            if verbose:
                self._print_stock_summary(results_df, ticker, horizon)
            
            return results_df
        
        except Exception as e:
            if verbose:
                print(f"  ❌ Backtest failed: {e}")
            return None
    
    def _prepare_features(self, df, nifty):
        """Prepare all features"""
        
        # Stock indicators
        df['RSI'] = EnhancedIndicators.get_rsi(df['Close'])
        df['ADX'] = EnhancedIndicators.get_adx(df['High'], df['Low'], df['Close'])
        df['NATR'] = EnhancedIndicators.get_natr(df['High'], df['Low'], df['Close'])
        df['OBV_Slope'] = EnhancedIndicators.get_obv_slope(df['Close'], df['Volume'])
        df['Dist_SMA'] = EnhancedIndicators.get_dist_sma(df['Close'])
        df['MACD'] = EnhancedIndicators.get_macd(df['Close'])
        df['ROC'] = EnhancedIndicators.get_roc(df['Close'])
        df['Vol_Ratio'] = EnhancedIndicators.get_volume_ratio(df['Volume'])
        df['BB_Position'] = EnhancedIndicators.get_bollinger_position(df['Close'])
        
        # Market features
        nifty_aligned = nifty.reindex(df.index, method='ffill')
        market_features = MarketContextFeatures.calculate_market_features(nifty_aligned['Close'])
        df = pd.concat([df, market_features], axis=1)
        
        stock_returns = df['Close'].pct_change(20)
        market_returns = nifty_aligned['Close'].pct_change(20)
        df['relative_strength'] = stock_returns - market_returns
        
        stock_ret = df['Close'].pct_change()
        market_ret = nifty_aligned['Close'].pct_change()
        df['beta'] = (stock_ret.rolling(60).cov(market_ret) / 
                     (market_ret.rolling(60).var() + 1e-8))
        
        df.dropna(inplace=True)
        return df
    
    def _predict_at_date(self, hist_data, horizon, ticker, pred_date):
        """Make prediction at specific date"""
        
        feature_cols = [
            'RSI', 'ADX', 'NATR', 'OBV_Slope', 'Dist_SMA',
            'MACD', 'ROC', 'Vol_Ratio', 'BB_Position',
            'market_strength', 'market_momentum', 'market_volatility', 'market_trend',
            'relative_strength', 'beta'
        ]
        
        last_seq = hist_data[feature_cols].tail(CONFIG['SEQ_LENGTH'])
        input_seq = self.model_loader.scaler.transform(last_seq)
        input_seq = np.array([input_seq])
        
        mean_forecast, std_forecast = self.model_loader.predict(input_seq)
        mean_forecast = mean_forecast[0]
        std_forecast = std_forecast[0]
        
        if horizon <= CONFIG['MAX_HORIZON']:
            pred_return = float(mean_forecast[horizon-1])
            uncertainty = float(std_forecast[horizon-1])
        else:
            pred_return = float(mean_forecast[-1])
            uncertainty = float(std_forecast[-1])
        
        # Calculate score (simplified from inference)
        current_natr = float(hist_data['NATR'].iloc[-1])
        expected_vol = current_natr * np.sqrt(horizon)
        z_score = pred_return / (expected_vol + uncertainty + 1e-8)
        
        from scipy.stats import norm
        score = float(norm.cdf(z_score) * 100)
        
        return pred_return, score, uncertainty
    
    def _get_actual_return(self, df, start_date, horizon):
        """Calculate actual return over horizon"""
        
        try:
            future_date = start_date + timedelta(days=horizon)
            
            # Find closest future date in data
            future_dates = df.index[df.index >= future_date]
            if len(future_dates) == 0:
                return None
            
            actual_future_date = future_dates[0]
            
            start_price = df.loc[start_date, 'Close']
            end_price = df.loc[actual_future_date, 'Close']
            
            return float((end_price - start_price) / start_price * 100)
        
        except:
            return None
    
    def _print_stock_summary(self, results_df, ticker, horizon):
        """Print summary for single stock"""
        
        n = len(results_df)
        dir_acc = results_df['direction_correct'].mean()
        
        corr, _ = pearsonr(results_df['predicted_return'], results_df['actual_return'])
        
        mae = np.mean(np.abs(results_df['actual_return'] - results_df['predicted_return']))
        rmse = np.sqrt(np.mean((results_df['actual_return'] - results_df['predicted_return'])**2))
        
        print(f"\n  📊 Results ({n} predictions):")
        print(f"     Direction Accuracy: {dir_acc:.1%}")
        print(f"     Correlation:        {corr:+.3f}")
        print(f"     MAE:                {mae:.2f}%")
        print(f"     RMSE:               {rmse:.2f}%")
        
        # Check if scores predict returns
        score_corr, _ = spearmanr(results_df['technical_score'], results_df['actual_return'])
        print(f"     Score→Return Corr:  {score_corr:+.3f}")
        
        if score_corr > 0.2:
            print(f"     ✅ Scores predict returns well")
        elif score_corr > 0.1:
            print(f"     🟡 Scores weakly predict returns")
        else:
            print(f"     ❌ Scores don't predict returns")

# ==========================================
# TRADING STRATEGY SIMULATOR
# ==========================================
class TradingStrategySimulator:
    """
    Simulate actual trading strategies based on scores
    """
    
    def __init__(self, backtest_results, config=BACKTEST_CONFIG):
        self.results = backtest_results
        self.config = config
    
    def strategy_long_only_threshold(self, score_threshold=60):
        """
        Strategy: Buy stocks with score > threshold, hold for horizon
        """
        
        print(f"\n{'='*80}")
        print(f" STRATEGY: Long-Only with Score Threshold = {score_threshold}")
        print(f"{'='*80}")
        
        # Filter signals
        signals = self.results[self.results['technical_score'] >= score_threshold].copy()
        
        if len(signals) == 0:
            print(f"  ❌ No signals above threshold")
            return None
        
        # Calculate returns
        signals['gross_return'] = signals['actual_return']
        signals['net_return'] = signals['actual_return'] - (self.config['TRANSACTION_COST'] * 100 * 2)
        
        # Statistics
        n_trades = len(signals)
        win_rate = (signals['actual_return'] > 0).mean()
        avg_return = signals['net_return'].mean()
        total_return = signals['net_return'].sum()
        sharpe = signals['net_return'].mean() / (signals['net_return'].std() + 1e-8) * np.sqrt(252/signals['horizon'].mean())
        
        max_dd = self._calculate_max_drawdown(signals['net_return'].values)
        
        print(f"\n  📊 Performance:")
        print(f"     Trades:           {n_trades}")
        print(f"     Win Rate:         {win_rate:.1%}")
        print(f"     Avg Return/Trade: {avg_return:+.2f}%")
        print(f"     Total Return:     {total_return:+.2f}%")
        print(f"     Sharpe Ratio:     {sharpe:.2f}")
        print(f"     Max Drawdown:     {max_dd:.2f}%")
        
        # Risk-adjusted return
        risk_adj_return = avg_return / (max_dd + 1e-8)
        print(f"     Risk-Adj Return:  {risk_adj_return:.2f}")
        
        return {
            'n_trades': n_trades,
            'win_rate': win_rate,
            'avg_return': avg_return,
            'total_return': total_return,
            'sharpe': sharpe,
            'max_drawdown': max_dd
        }
    
    def strategy_top_n_stocks(self, top_n=5):
        """
        Strategy: Buy top N stocks by score at each rebalance
        """
        
        print(f"\n{'='*80}")
        print(f" STRATEGY: Top-{top_n} Portfolio (Rebalanced)")
        print(f"{'='*80}")
        
        # Group by date
        dates = sorted(self.results['date'].unique())
        
        portfolio_returns = []
        
        for date in dates:
            # Get predictions for this date
            day_signals = self.results[self.results['date'] == date].copy()
            
            # Select top N
            top_stocks = day_signals.nlargest(top_n, 'technical_score')
            
            if len(top_stocks) > 0:
                # Equal weight portfolio
                avg_return = top_stocks['actual_return'].mean()
                portfolio_returns.append(avg_return)
        
        if len(portfolio_returns) == 0:
            print(f"  ❌ No valid portfolio periods")
            return None
        
        portfolio_returns = np.array(portfolio_returns)
        
        # Apply transaction costs
        net_returns = portfolio_returns - (self.config['TRANSACTION_COST'] * 100 * 2)
        
        avg_return = net_returns.mean()
        total_return = net_returns.sum()
        sharpe = net_returns.mean() / (net_returns.std() + 1e-8) * np.sqrt(252/self.results['horizon'].mean())
        max_dd = self._calculate_max_drawdown(net_returns)
        
        print(f"\n  📊 Performance:")
        print(f"     Rebalances:       {len(portfolio_returns)}")
        print(f"     Avg Return/Period:{avg_return:+.2f}%")
        print(f"     Total Return:     {total_return:+.2f}%")
        print(f"     Sharpe Ratio:     {sharpe:.2f}")
        print(f"     Max Drawdown:     {max_dd:.2f}%")
        
        return {
            'n_periods': len(portfolio_returns),
            'avg_return': avg_return,
            'total_return': total_return,
            'sharpe': sharpe,
            'max_drawdown': max_dd
        }
    
    def strategy_long_short(self, long_threshold=60, short_threshold=40):
        """
        Strategy: Long high scores, short low scores
        """
        
        print(f"\n{'='*80}")
        print(f" STRATEGY: Long-Short (Long>{long_threshold}, Short<{short_threshold})")
        print(f"{'='*80}")
        
        long_signals = self.results[self.results['technical_score'] >= long_threshold].copy()
        short_signals = self.results[self.results['technical_score'] <= short_threshold].copy()
        
        if len(long_signals) == 0 and len(short_signals) == 0:
            print(f"  ❌ No signals")
            return None
        
        # Long returns
        long_returns = long_signals['actual_return'] - (self.config['TRANSACTION_COST'] * 100 * 2)
        
        # Short returns (profit when stock goes down)
        short_returns = -short_signals['actual_return'] - (self.config['TRANSACTION_COST'] * 100 * 2)
        
        all_returns = pd.concat([long_returns, short_returns])
        
        n_long = len(long_signals)
        n_short = len(short_signals)
        avg_return = all_returns.mean()
        total_return = all_returns.sum()
        sharpe = all_returns.mean() / (all_returns.std() + 1e-8) * np.sqrt(252/self.results['horizon'].mean())
        
        print(f"\n  📊 Performance:")
        print(f"     Long Positions:   {n_long}")
        print(f"     Short Positions:  {n_short}")
        print(f"     Avg Return/Trade: {avg_return:+.2f}%")
        print(f"     Total Return:     {total_return:+.2f}%")
        print(f"     Sharpe Ratio:     {sharpe:.2f}")
        
        return {
            'n_long': n_long,
            'n_short': n_short,
            'avg_return': avg_return,
            'total_return': total_return,
            'sharpe': sharpe
        }
    
    def _calculate_max_drawdown(self, returns):
        """Calculate maximum drawdown"""
        cumulative = np.cumsum(returns)
        running_max = np.maximum.accumulate(cumulative)
        drawdown = cumulative - running_max
        return float(np.min(drawdown)) if len(drawdown) > 0 else 0.0

# ==========================================
# BENCHMARK COMPARISON
# ==========================================
class BenchmarkComparison:
    """Compare strategy to benchmarks"""
    
    @staticmethod
    def compare_to_buy_and_hold(results_df, benchmark='^NSEI'):
        """Compare to buy-and-hold NIFTY"""
        
        print(f"\n{'='*80}")
        print(f" BENCHMARK COMPARISON: Buy & Hold {benchmark}")
        print(f"{'='*80}")
        
        # Get benchmark data
        start_date = results_df['date'].min()
        end_date = results_df['date'].max() + timedelta(days=180)
        
        bench_data = yf.download(benchmark, start=start_date, end=end_date, progress=False)
        if isinstance(bench_data.columns, pd.MultiIndex):
            bench_data.columns = bench_data.columns.get_level_values(0)
        
        # Calculate buy-and-hold return
        bench_start = bench_data['Close'].iloc[0]
        bench_end = bench_data['Close'].iloc[-1]
        bench_return = (bench_end - bench_start) / bench_start * 100
        
        days = (end_date - start_date).days
        bench_annual = (bench_return / days) * 365
        
        print(f"\n  📊 Buy & Hold {benchmark}:")
        print(f"     Period Return:    {bench_return:+.2f}%")
        print(f"     Annualized:       {bench_annual:+.2f}%")
        
        return bench_return, bench_annual

# ==========================================
# MAIN BACKTESTING PIPELINE
# ==========================================
def run_comprehensive_backtest():
    """Run complete backtesting suite"""
    
    print("\n" + "="*80)
    print(" 🚀 COMPREHENSIVE BACKTESTING SYSTEM")
    print("="*80)
    
    print(f"\n⚙️  Configuration:")
    print(f"   Test Start:       {BACKTEST_CONFIG['TEST_START_DATE']}")
    print(f"   Initial Capital:  ₹{BACKTEST_CONFIG['INITIAL_CAPITAL']:,}")
    print(f"   Transaction Cost: {BACKTEST_CONFIG['TRANSACTION_COST']*100:.2f}%")
    print(f"   Rebalance Freq:   {BACKTEST_CONFIG['REBALANCE_FREQUENCY']} days")
    
    # Load model
    print(f"\n🔧 Loading model...")
    model_loader = ModelLoader()
    
    # Create backtester
    backtester = ModelBacktester(model_loader)
    
    # Test stocks
    test_stocks = [
        'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
        'HINDUNILVR.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'ITC.NS', 'KOTAKBANK.NS'
    ]
    
    # Test horizons
    test_horizons = [20, 60, 120]
    
    all_results = []
    
    print(f"\n{'='*80}")
    print(f" PHASE 1: GENERATING PREDICTIONS")
    print(f"{'='*80}")
    
    for horizon in test_horizons:
        print(f"\n--- Testing Horizon: {horizon} days ---")
        
        for stock in test_stocks:
            result = backtester.backtest_single_stock(stock, horizon, verbose=False)
            if result is not None:
                all_results.append(result)
                print(f"  ✓ {stock:15s} - {len(result)} predictions")
    
    if not all_results:
        print("\n❌ No backtest results generated. Check data availability.")
        return
    
    # Combine all results
    combined_results = pd.concat(all_results, ignore_index=True)
    
    print(f"\n✅ Total Predictions: {len(combined_results):,}")
    
    # Overall correlation check
    print(f"\n{'='*80}")
    print(f" PHASE 2: SCORE VALIDATION")
    print(f"{'='*80}")
    
    overall_corr, _ = spearmanr(combined_results['technical_score'], 
                                combined_results['actual_return'])
    
    print(f"\n  📊 Overall Score → Return Correlation: {overall_corr:+.3f}")
    
    if overall_corr > 0.2:
        print(f"     ✅ STRONG - Scores reliably predict returns")
    elif overall_corr > 0.1:
        print(f"     🟡 MODERATE - Scores weakly predict returns")
    elif overall_corr > 0:
        print(f"     🟠 WEAK - Minimal predictive power")
    else:
        print(f"     ❌ NEGATIVE - Scores anti-predict returns!")
    
    # By horizon
    print(f"\n  Score→Return Correlation by Horizon:")
    for horizon in test_horizons:
        h_data = combined_results[combined_results['horizon'] == horizon]
        if len(h_data) > 5:
            h_corr, _ = spearmanr(h_data['technical_score'], h_data['actual_return'])
            print(f"     {horizon:3d} days: {h_corr:+.3f}")
    
    # Test trading strategies
    print(f"\n{'='*80}")
    print(f" PHASE 3: STRATEGY SIMULATION")
    print(f"{'='*80}")
    
    simulator = TradingStrategySimulator(combined_results)
    
    # Strategy 1: Threshold-based
    print(f"\n--- Strategy 1: Threshold-Based Long ---")
    for threshold in [55, 60, 65]:
        simulator.strategy_long_only_threshold(threshold)
    
    # Strategy 2: Top-N portfolio
    print(f"\n--- Strategy 2: Top-N Portfolio ---")
    for n in [3, 5, 10]:
        simulator.strategy_top_n_stocks(n)
    
    # Strategy 3: Long-Short
    print(f"\n--- Strategy 3: Long-Short ---")
    simulator.strategy_long_short(long_threshold=60, short_threshold=40)
    
    # Benchmark comparison
    print(f"\n{'='*80}")
    print(f" PHASE 4: BENCHMARK COMPARISON")
    print(f"{'='*80}")
    
    BenchmarkComparison.compare_to_buy_and_hold(combined_results)
    
    # Final recommendations
    print(f"\n{'='*80}")
    print(f" 🎯 FINAL VERDICT")
    print(f"{'='*80}")
    
    # Calculate key metrics
    dir_acc = combined_results['direction_correct'].mean()
    avg_return_when_positive = combined_results[
        combined_results['technical_score'] > 60
    ]['actual_return'].mean()
    
    print(f"\n  Key Findings:")
    print(f"     Overall Direction Accuracy: {dir_acc:.1%}")
    print(f"     Avg Return (Score>60):      {avg_return_when_positive:+.2f}%")
    print(f"     Score Predictiveness:       {overall_corr:+.3f}")
    
    if dir_acc > 0.6 and overall_corr > 0.15:
        verdict = "✅ PROFITABLE - Model shows edge, proceed with caution"
    elif dir_acc > 0.55 and overall_corr > 0.1:
        verdict = "🟡 MARGINAL - Edge exists but small, need risk management"
    else:
        verdict = "❌ NOT READY - Model needs improvement before trading"
    
    print(f"\n  Verdict: {verdict}")
    
    print(f"\n  Next Steps:")
    if dir_acc > 0.55:
        print(f"     1. Paper trade for 3 months")
        print(f"     2. Start with small capital (5-10%)")
        print(f"     3. Focus on {120}-day horizon (best performance)")
        print(f"     4. Use strict stop-losses (8-10%)")
        print(f"     5. Diversify across 10+ positions")
    else:
        print(f"     1. Retrain with more data or features")
        print(f"     2. Consider ensemble improvements")
        print(f"     3. Test different loss functions")
        print(f"     4. Add more technical indicators")
    
    print("\n" + "="*80 + "\n")
    
    # Save results
    output_file = 'backtest_results.csv'
    combined_results.to_csv(output_file, index=False)
    print(f"💾 Results saved to: {output_file}\n")

if __name__ == "__main__":
    run_comprehensive_backtest()


 🚀 COMPREHENSIVE BACKTESTING SYSTEM

⚙️  Configuration:
   Test Start:       2024-01-01
   Initial Capital:  ₹1,000,000
   Transaction Cost: 0.20%
   Rebalance Freq:   20 days

🔧 Loading model...
✓ Loaded scaler: scaler_v2.pkl
✓ Detected ensemble in ensemble_models
Loading 5 ensemble models...
  ✓ Model 1/5
  ✓ Model 2/5
  ✓ Model 3/5
  ✓ Model 4/5
  ✓ Model 5/5
✓ Successfully loaded 5 ensemble models
  Weights: ['0.207', '0.208', '0.195', '0.195', '0.195']

 PHASE 1: GENERATING PREDICTIONS

--- Testing Horizon: 20 days ---
  ✓ RELIANCE.NS     - 15 predictions
  ✓ TCS.NS          - 15 predictions
  ✓ HDFCBANK.NS     - 15 predictions
  ✓ INFY.NS         - 15 predictions
  ✓ ICICIBANK.NS    - 15 predictions
  ✓ HINDUNILVR.NS   - 15 predictions
  ✓ SBIN.NS         - 15 predictions
  ✓ BHARTIARTL.NS   - 15 predictions
  ✓ ITC.NS          - 15 predictions
  ✓ KOTAKBANK.NS    - 15 predictions

--- Testing Horizon: 60 days ---
  ✓ RELIANCE.NS     - 9 predictions
  ✓ TCS.NS          - 9 predi

In [6]:
"""
NIFTY-50 Multi-Horizon Stock Predictor - Inference System v2.0 (FIXED)

Fixes:
- Custom loss function registration
- Proper model loading with custom objects
- Better error handling

Author: Enhanced Inference Pipeline
Date: 2025
"""

import os
import glob
import joblib
import yfinance as yf
import pandas as pd
import numpy as np
import tensorflow as tf
import warnings
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

warnings.filterwarnings("ignore")

# ==========================================
# CUSTOM LOSS (Must be defined before loading models)
# ==========================================
class DirectionAwareLoss(tf.keras.losses.Loss):
    """Penalize wrong direction predictions more - Compatible with TF 2.x"""
    def __init__(self, direction_weight=0.3, name='direction_aware_loss', reduction='sum_over_batch_size'):
        super().__init__(name=name, reduction=reduction)
        self.direction_weight = direction_weight
        
    def call(self, y_true, y_pred):
        # MSE component
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        
        # Direction penalty
        direction_penalty = tf.reduce_mean(
            tf.cast(tf.sign(y_true) != tf.sign(y_pred), tf.float32)
        )
        
        return (1 - self.direction_weight) * mse + self.direction_weight * direction_penalty * 100
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'direction_weight': self.direction_weight
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)

# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    'SEQ_LENGTH': 60,
    'MAX_HORIZON': 180,
    'SCALER_PATH': 'scaler_v2.pkl',
    'ENSEMBLE_DIR': 'ensemble_models',
    'ENSEMBLE_INFO_PATH': 'nifty50_integrated_ensemble_info.pkl',
    'SINGLE_MODEL_PATH': 'nifty50_integrated_single.keras',
    'USE_ENSEMBLE': True,  # Auto-detect if ensemble exists
}

# ==========================================
# INDICATORS (Must match training)
# ==========================================
class EnhancedIndicators:
    @staticmethod
    def get_rsi(series, period=14):
        delta = series.diff()
        gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
        loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))

    @staticmethod
    def get_adx(high, low, close, period=14):
        plus_dm = high.diff()
        minus_dm = low.diff()
        plus_dm = plus_dm.where(plus_dm > 0, 0)
        minus_dm = minus_dm.where(minus_dm < 0, 0).abs()
        
        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low - close.shift(1)).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        
        atr = tr.ewm(alpha=1/period, adjust=False).mean()
        plus_di = 100 * (plus_dm.ewm(alpha=1/period, adjust=False).mean() / atr)
        minus_di = 100 * (minus_dm.ewm(alpha=1/period, adjust=False).mean() / atr)
        dx = (abs(plus_di - minus_di) / (plus_di + minus_di + 1e-8)) * 100
        return dx.ewm(alpha=1/period, adjust=False).mean()

    @staticmethod
    def get_natr(high, low, close, period=14):
        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low - close.shift(1)).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1/period, adjust=False).mean()
        return (atr / close) * 100

    @staticmethod
    def get_obv_slope(close, volume, period=14):
        obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
        return obv.diff(period)

    @staticmethod
    def get_dist_sma(close, period=50):
        sma = close.rolling(period).mean()
        return (close - sma) / sma
    
    @staticmethod
    def get_macd(close, fast=12, slow=26, signal=9):
        ema_fast = close.ewm(span=fast, adjust=False).mean()
        ema_slow = close.ewm(span=slow, adjust=False).mean()
        macd = ema_fast - ema_slow
        macd_signal = macd.ewm(span=signal, adjust=False).mean()
        return macd - macd_signal
    
    @staticmethod
    def get_roc(close, period=10):
        return ((close - close.shift(period)) / close.shift(period)) * 100
    
    @staticmethod
    def get_volume_ratio(volume, period=20):
        vol_ma = volume.rolling(period).mean()
        return volume / vol_ma
    
    @staticmethod
    def get_bollinger_position(close, period=20, std=2):
        sma = close.rolling(period).mean()
        rolling_std = close.rolling(period).std()
        upper = sma + (rolling_std * std)
        lower = sma - (rolling_std * std)
        # Avoid division by zero
        band_width = upper - lower
        band_width = band_width.replace(0, np.nan)
        return (close - lower) / band_width

# ==========================================
# MARKET CONTEXT
# ==========================================
class MarketContextFeatures:
    @staticmethod
    def calculate_market_features(nifty_close):
        features = pd.DataFrame(index=nifty_close.index)
        
        for period in [20, 50, 200]:
            ma = nifty_close.rolling(period).mean()
            features[f'above_ma_{period}'] = (nifty_close > ma).astype(float)
        
        features['market_strength'] = features[[f'above_ma_{p}' for p in [20, 50, 200]]].mean(axis=1)
        features['market_momentum'] = nifty_close.pct_change(20) * 100
        
        returns = nifty_close.pct_change()
        features['market_volatility'] = returns.rolling(20).std() * np.sqrt(252) * 100
        
        # Fix division by zero
        rolling_std = returns.rolling(20).std()
        features['market_trend'] = returns.rolling(20).mean() / (rolling_std + 1e-8)
        
        return features[['market_strength', 'market_momentum', 'market_volatility', 'market_trend']]

# ==========================================
# MODEL LOADER
# ==========================================
class ModelLoader:
    """Load either ensemble or single model with proper custom object handling"""
    
    def __init__(self):
        self.scaler = None
        self.models = []
        self.weights = []
        self.is_ensemble = False
        self._load_models()
    
    def _get_custom_objects(self):
        """Return dictionary of custom objects for model loading"""
        return {
            'DirectionAwareLoss': DirectionAwareLoss
        }
    
    def _load_models(self):
        """Auto-detect and load appropriate model type"""
        
        # Load scaler
        if not os.path.exists(CONFIG['SCALER_PATH']):
            raise FileNotFoundError(f"Scaler not found: {CONFIG['SCALER_PATH']}")
        
        self.scaler = joblib.load(CONFIG['SCALER_PATH'])
        print(f"✓ Loaded scaler: {CONFIG['SCALER_PATH']}")
        
        # Try to load ensemble first
        if os.path.exists(CONFIG['ENSEMBLE_DIR']) and os.path.exists(CONFIG['ENSEMBLE_INFO_PATH']):
            print(f"✓ Detected ensemble in {CONFIG['ENSEMBLE_DIR']}")
            self._load_ensemble()
            self.is_ensemble = True
        
        # Fallback to single model
        elif os.path.exists(CONFIG['SINGLE_MODEL_PATH']):
            print(f"✓ Loading single model: {CONFIG['SINGLE_MODEL_PATH']}")
            model = load_model(CONFIG['SINGLE_MODEL_PATH'], 
                             custom_objects=self._get_custom_objects(),
                             compile=False)
            self.models = [model]
            self.weights = [1.0]
            self.is_ensemble = False
        
        else:
            raise FileNotFoundError("No trained model found. Please run training first.")
    
    def _load_ensemble(self):
        """Load all ensemble models with custom objects"""
        ensemble_info = joblib.load(CONFIG['ENSEMBLE_INFO_PATH'])
        self.weights = np.array(ensemble_info['weights'])
        
        model_files = sorted(glob.glob(os.path.join(CONFIG['ENSEMBLE_DIR'], 'model_*.keras')))
        
        print(f"Loading {len(model_files)} ensemble models...")
        for i, model_file in enumerate(model_files, 1):
            try:
                # Try with custom objects first
                try:
                    model = load_model(model_file, 
                                     custom_objects=self._get_custom_objects(),
                                     compile=False)
                except:
                    # If that fails, try loading with safe_mode
                    print(f"  ⚠ Trying safe_mode for model {i}...")
                    model = tf.keras.models.load_model(model_file, compile=False)
                
                self.models.append(model)
                print(f"  ✓ Model {i}/{len(model_files)}")
            except Exception as e:
                print(f"  ✗ Failed to load {model_file}: {e}")
                raise
        
        print(f"✓ Successfully loaded {len(self.models)} ensemble models")
        print(f"  Weights: {[f'{w:.3f}' for w in self.weights]}")
    
    def predict(self, X):
        """Predict with loaded model(s)"""
        if self.is_ensemble:
            predictions = []
            for model in self.models:
                pred = model.predict(X, verbose=0)
                predictions.append(pred)
            
            predictions = np.array(predictions)
            weighted_pred = np.average(predictions, axis=0, weights=self.weights)
            std_pred = np.std(predictions, axis=0)
            
            return weighted_pred, std_pred
        else:
            pred = self.models[0].predict(X, verbose=0)
            return pred, np.zeros_like(pred)  # No uncertainty for single model

# ==========================================
# MARKET BASELINE
# ==========================================
class MarketBaseline:
    @staticmethod
    def get_nifty_baseline():
        try:
            nifty = yf.download('^NSEI', period='10y', interval='1d', progress=False)
            if len(nifty) < 252:
                return {'annual_return': 12.0, 'annual_vol': 15.0}
            
            returns = nifty['Close'].pct_change().dropna()
            annual_return = (nifty['Close'].pct_change(252).mean() * 100)
            annual_vol = (returns.std() * np.sqrt(252) * 100)
            
            return {'annual_return': annual_return, 'annual_vol': annual_vol}
        except:
            return {'annual_return': 12.0, 'annual_vol': 15.0}

# ==========================================
# FEATURE PREPARATION
# ==========================================
def prepare_features_for_inference(ticker, nifty_df=None):
    """Prepare features matching training pipeline"""
    
    # Download stock data
    df = yf.download(ticker, period="2y", interval="1d", progress=False)
    
    if len(df) < 150:
        raise ValueError("Insufficient historical data")
    
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Calculate stock indicators
    df['RSI'] = EnhancedIndicators.get_rsi(df['Close'])
    df['ADX'] = EnhancedIndicators.get_adx(df['High'], df['Low'], df['Close'])
    df['NATR'] = EnhancedIndicators.get_natr(df['High'], df['Low'], df['Close'])
    df['OBV_Slope'] = EnhancedIndicators.get_obv_slope(df['Close'], df['Volume'])
    df['Dist_SMA'] = EnhancedIndicators.get_dist_sma(df['Close'])
    df['MACD'] = EnhancedIndicators.get_macd(df['Close'])
    df['ROC'] = EnhancedIndicators.get_roc(df['Close'])
    df['Vol_Ratio'] = EnhancedIndicators.get_volume_ratio(df['Volume'])
    df['BB_Position'] = EnhancedIndicators.get_bollinger_position(df['Close'])
    
    # Add market context if NIFTY data provided
    if nifty_df is not None:
        nifty_aligned = nifty_df.reindex(df.index, method='ffill')
        
        market_features = MarketContextFeatures.calculate_market_features(nifty_aligned['Close'])
        df = pd.concat([df, market_features], axis=1)
        
        stock_returns = df['Close'].pct_change(20)
        market_returns = nifty_aligned['Close'].pct_change(20)
        df['relative_strength'] = stock_returns - market_returns
        
        stock_ret = df['Close'].pct_change()
        market_ret = nifty_aligned['Close'].pct_change()
        df['beta'] = (
            stock_ret.rolling(60).cov(market_ret) / 
            (market_ret.rolling(60).var() + 1e-8)
        )
    
    df.dropna(inplace=True)
    
    return df

# ==========================================
# SCORING ENGINE
# ==========================================
def analyze_technical_score(ticker, duration_days, model_loader=None, verbose=True):
    """
    Complete technical analysis with ensemble support
    """
    
    result = {
        'ticker': ticker,
        'duration': duration_days,
        'technical_score': 50,
        'predicted_return': 0.0,
        'uncertainty': 0.0,
        'confidence': 0.0,
        'signal': 'NEUTRAL',
        'status': 'FAIL',
        'message': '',
        'indicators': {},
        'is_ensemble': False
    }
    
    try:
        # Load models if not provided
        if model_loader is None:
            model_loader = ModelLoader()
        
        result['is_ensemble'] = model_loader.is_ensemble
        
        # Download NIFTY for market context
        try:
            nifty_df = yf.download('^NSEI', period='2y', interval='1d', progress=False)
            if isinstance(nifty_df.columns, pd.MultiIndex):
                nifty_df.columns = nifty_df.columns.get_level_values(0)
        except Exception as e:
            result['message'] = f"Failed to download NIFTY data: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        # Prepare features
        try:
            df = prepare_features_for_inference(ticker, nifty_df)
        except Exception as e:
            result['message'] = f"Feature preparation failed: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        if len(df) < CONFIG['SEQ_LENGTH']:
            result['message'] = "Insufficient sequence length"
            return result
        
        # Store current indicators
        try:
            result['indicators'] = {
                'RSI': round(float(df['RSI'].iloc[-1]), 2),
                'ADX': round(float(df['ADX'].iloc[-1]), 2),
                'NATR': round(float(df['NATR'].iloc[-1]), 2),
                'MACD': round(float(df['MACD'].iloc[-1]), 4),
                'ROC': round(float(df['ROC'].iloc[-1]), 2),
                'BB_Position': round(float(df['BB_Position'].iloc[-1]), 2)
            }
            
            if 'market_strength' in df.columns:
                result['indicators']['Market_Strength'] = round(float(df['market_strength'].iloc[-1]), 2)
                result['indicators']['Beta'] = round(float(df['beta'].iloc[-1]), 2)
        except Exception as e:
            result['message'] = f"Indicator extraction failed: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        # Prepare input sequence
        feature_cols = [
            'RSI', 'ADX', 'NATR', 'OBV_Slope', 'Dist_SMA',
            'MACD', 'ROC', 'Vol_Ratio', 'BB_Position'
        ]
        
        if 'market_strength' in df.columns:
            feature_cols.extend([
                'market_strength', 'market_momentum', 'market_volatility', 'market_trend',
                'relative_strength', 'beta'
            ])
        
        try:
            last_seq = df[feature_cols].tail(CONFIG['SEQ_LENGTH'])
            
            # Scale features
            input_seq = model_loader.scaler.transform(last_seq)
            input_seq = np.array([input_seq])
        except Exception as e:
            result['message'] = f"Sequence preparation failed: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        # Predict
        try:
            mean_forecast, std_forecast = model_loader.predict(input_seq)
            mean_forecast = mean_forecast[0]
            std_forecast = std_forecast[0]
        except Exception as e:
            result['message'] = f"Prediction failed: {e}"
            if verbose:
                print(f"❌ {result['message']}")
            return result
        
        # Get prediction for requested horizon
        eff_duration = min(duration_days, CONFIG['MAX_HORIZON'])
        raw_return = float(mean_forecast[eff_duration - 1])
        prediction_std = float(std_forecast[eff_duration - 1])
        
        # === SCORING LOGIC ===
        
        # Market baseline
        market_baseline = MarketBaseline.get_nifty_baseline()
        time_factor = eff_duration / 365.0
        period_baseline_return = float(market_baseline['annual_return'] * time_factor)
        period_baseline_vol = float(market_baseline['annual_vol'] * np.sqrt(time_factor))
        
        # Stock volatility
        current_natr = float(df['NATR'].iloc[-1])
        expected_vol_move = float(current_natr * np.sqrt(eff_duration))
        
        # Calculate Z-score
        total_uncertainty = float(np.sqrt(expected_vol_move**2 + period_baseline_vol**2 + prediction_std**2))
        
        if total_uncertainty > 0:
            final_z = float((raw_return - period_baseline_return) / total_uncertainty)
        else:
            final_z = 0.0
        
        # Confidence adjustment
        current_adx = float(df['ADX'].iloc[-1])
        current_rsi = float(df['RSI'].iloc[-1])
        
        confidence = 1.0
        
        if current_adx < 20:
            confidence *= 0.7
        elif current_adx > 40:
            confidence *= 1.1
        
        if current_rsi > 70 and raw_return > 0:
            confidence *= 0.8
        elif current_rsi < 30 and raw_return < 0:
            confidence *= 0.8
        
        if prediction_std > 5:
            confidence *= 0.85
        
        # Ensemble bonus
        if model_loader.is_ensemble:
            confidence *= 1.05  # Slight boost for ensemble
        
        confidence = min(confidence, 1.2)
        
        # Convert to score
        base_score = norm.cdf(final_z) * 100
        final_score = 50 + (base_score - 50) * confidence
        final_score = np.clip(final_score, 0, 100)
        
        # Determine signal
        if final_score > 75:
            signal = 'STRONG BUY'
        elif final_score > 65:
            signal = 'BUY'
        elif final_score > 55:
            signal = 'WEAK BUY'
        elif final_score > 45:
            signal = 'NEUTRAL'
        elif final_score > 35:
            signal = 'WEAK SELL'
        elif final_score > 25:
            signal = 'SELL'
        else:
            signal = 'STRONG SELL'
        
        # Update result
        result.update({
            'technical_score': round(float(final_score), 2),
            'predicted_return': round(float(raw_return), 2),
            'uncertainty': round(float(prediction_std), 2),
            'confidence': round(float(confidence), 2),
            'signal': signal,
            'status': 'SUCCESS',
            'z_score': round(float(final_z), 3),
            'market_baseline': round(float(period_baseline_return), 2)
        })
    
    except Exception as e:
        import traceback
        result['message'] = f"Error: {str(e)}"
        if verbose:
            print(f"❌ {result['message']}")
            print(traceback.format_exc())
    
    return result

# ==========================================
# REPORTING
# ==========================================
def print_analysis_report(result):
    """Enhanced report with ensemble info"""
    
    print("\n" + "="*80)
    print(f" TECHNICAL ANALYSIS REPORT: {result['ticker']}")
    print("="*80)
    
    if result['status'] != 'SUCCESS':
        print(f"\n❌ Analysis Failed: {result['message']}")
        print("="*80)
        return
    
    # Model info
    model_type = "🎯 ENSEMBLE" if result['is_ensemble'] else "📊 SINGLE MODEL"
    print(f"\n{model_type}")
    
    print(f"\n📊 PREDICTION ({result['duration']}-Day Horizon)")
    print(f"   Expected Return:    {result['predicted_return']:+.2f}%")
    print(f"   Uncertainty (±):    {result['uncertainty']:.2f}%")
    print(f"   Market Baseline:    {result['market_baseline']:+.2f}%")
    print(f"   Z-Score:           {result['z_score']:+.3f}")
    
    print(f"\n🎯 SCORING")
    print(f"   Technical Score:    {result['technical_score']:.1f} / 100")
    print(f"   Model Confidence:   {result['confidence']:.2f}x")
    
    score = result['technical_score']
    if score > 75:
        emoji = "🟢🟢"
    elif score > 65:
        emoji = "🟢"
    elif score < 25:
        emoji = "🔴🔴"
    elif score < 35:
        emoji = "🔴"
    else:
        emoji = "⚪"
    
    print(f"   Signal:             {emoji} {result['signal']}")
    
    print(f"\n📈 CURRENT INDICATORS")
    for name, value in result['indicators'].items():
        print(f"   {name:18s}: {value}")
    
    print("\n" + "="*80 + "\n")

# ==========================================
# BATCH ANALYSIS
# ==========================================
def analyze_portfolio(tickers, duration_days, model_loader=None):
    """Analyze multiple stocks efficiently"""
    
    if model_loader is None:
        model_loader = ModelLoader()
    
    print(f"\n{'='*80}")
    print(f" PORTFOLIO ANALYSIS - {duration_days} Day Horizon")
    print(f" Model: {'Ensemble' if model_loader.is_ensemble else 'Single'}")
    print(f"{'='*80}\n")
    
    results = []
    
    for ticker in tickers:
        print(f"Analyzing {ticker:15s}...", end=' ')
        result = analyze_technical_score(ticker, duration_days, model_loader, verbose=False)
        
        if result['status'] == 'SUCCESS':
            print(f"✓ Score: {result['technical_score']:.1f}")
            results.append(result)
        else:
            print(f"✗ Failed")
    
    # Sort by score
    results.sort(key=lambda x: x['technical_score'], reverse=True)
    
    # Print ranking
    print(f"\n{'='*80}")
    print(f" RANKING BY TECHNICAL SCORE")
    print(f"{'='*80}")
    print(f"{'Rank':<6} {'Ticker':<15} {'Score':<10} {'Return':<12} {'Signal':<15}")
    print("-"*80)
    
    for i, res in enumerate(results, 1):
        print(f"{i:<6} {res['ticker']:<15} {res['technical_score']:<10.1f} "
              f"{res['predicted_return']:>+6.2f}%     {res['signal']:<15}")
    
    print("="*80 + "\n")
    
    return results

# ==========================================
# MAIN - USER INPUT MODE
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*80)
    print(" NIFTY-50 TECHNICAL SCORE INFERENCE")
    print("="*80)

    # Initialize model loader
    try:
        model_loader = ModelLoader()
    except Exception as e:
        print(f"\n❌ Failed to load models: {e}")
        print("Please ensure models and scaler exist.")
        exit(1)

    # ---- User Input ----
    print("\nEnter the stock ticker and horizon to get the investment score.")
    print("Example NSE tickers: RELIANCE.NS, TCS.NS, INFY.NS, HDFCBANK.NS, SBIN.NS")
    print("---------------------------------------------------------------")

    ticker = input("Enter company ticker (e.g., RELIANCE.NS): ").strip().upper()
    duration = input("Enter investment duration in days (e.g., 30): ").strip()

    # Validate duration
    try:
        duration = int(duration)
        if duration <= 0:
            raise ValueError
    except:
        print("\n❌ Invalid duration. Must be a positive integer.")
        exit(1)

    print("\nRunning analysis... Please wait.\n")

    # ---- Run analysis ----
    result = analyze_technical_score(ticker, duration, model_loader)

    # ---- Print final output ----
    print_analysis_report(result)



 NIFTY-50 TECHNICAL SCORE INFERENCE
✓ Loaded scaler: scaler_v2.pkl
✓ Detected ensemble in ensemble_models
Loading 5 ensemble models...
  ✓ Model 1/5
  ✓ Model 2/5
  ✓ Model 3/5
  ✓ Model 4/5
  ✓ Model 5/5
✓ Successfully loaded 5 ensemble models
  Weights: ['0.207', '0.208', '0.195', '0.195', '0.195']

Enter the stock ticker and horizon to get the investment score.
Example NSE tickers: RELIANCE.NS, TCS.NS, INFY.NS, HDFCBANK.NS, SBIN.NS
---------------------------------------------------------------

Running analysis... Please wait.


 TECHNICAL ANALYSIS REPORT: SBIN.NS

🎯 ENSEMBLE

📊 PREDICTION (150-Day Horizon)
   Expected Return:    +11.47%
   Uncertainty (±):    4.82%
   Market Baseline:    +5.97%
   Z-Score:           +0.274

🎯 SCORING
   Technical Score:    62.5 / 100
   Model Confidence:   1.16x
   Signal:             ⚪ WEAK BUY

📈 CURRENT INDICATORS
   RSI               : 66.15
   ADX               : 47.9
   NATR              : 1.34
   MACD              : -2.2966
   ROC         